# TELEPATI 8.0: AgriData Intelligence Race
## Deteksi Penyakit Tanaman Padi Berbasis *Object Detection*

Laporan ini kami susun sebagai dokumen penelitian, bukan sebagai kumpulan
sel kode. Kami menyarankan pembaca mengikutinya dari atas ke bawah, karena
setiap bagian dibangun di atas temuan bagian sebelumnya: kami mulai dari
masalah yang ingin diselesaikan, lalu memeriksa kondisi data, menarik
implikasinya terhadap pemodelan, menguji keputusan lewat eksperimen
terkontrol, membaca hasilnya, membedah kesalahan yang masih terjadi, dan
menutupnya dengan keterbatasan yang kami akui secara terbuka.

Sebelum masuk ke isi, kami perlu menegaskan kepatuhan terhadap aturan
kompetisi. Model kami bangun sepenuhnya dari definisi arsitektur `.yaml`
dengan `pretrained=False`, sehingga tidak ada *external pretrained weights*
dalam bentuk apa pun. Kami tidak memakai dataset di luar dataset resmi, dan
tidak memproses dataset menggunakan LLM maupun API eksternal. Pemetaan 11
kelas canonical kami verifikasi terhadap dataset aktual tanpa menyisakan
kategori yang tidak terpetakan. Seluruh penyemaian bersifat deterministik,
dan `YOLO_OFFLINE=1` kami aktifkan pada setiap skrip pelatihan maupun
evaluasi agar upaya pengunduhan bobot gagal secara terlihat, bukan lolos
diam-diam.

Agar notebook ini dapat diperiksa tanpa menunggu proses pelatihan berjam-jam,
secara bawaan kami menjalankannya pada mode evaluasi (`SKIP_TRAINING = True`)
yang memuat bobot final yang sudah dilatih. Mode reproduksi penuh tetap
tersedia dan kami jelaskan pada Bagian 13. Logika analisisnya sendiri kami
simpan pada paket `src/agridata/` dan skrip pada `scripts/`, sehingga
notebook ini memanggil fungsi yang sudah diuji, bukan menyalin ulang kode.

## 1. Latar Belakang dan Tujuan

Padi merupakan komoditas pangan utama, dan penyakit pada daun serta malai
padi dapat menurunkan hasil panen secara signifikan. Di lapangan, penyakit
umumnya diidentifikasi secara manual melalui pengamatan visual petani atau
penyuluh. Cara itu menuntut pengalaman, memakan waktu, dan sulit diskalakan
untuk area tanam yang luas.

Dari situlah kami melihat ruang bagi pendekatan berbasis *computer vision*:
satu model dapat memeriksa ribuan citra dalam waktu singkat. Kelayakan
pembelajaran mendalam untuk mengenali penyakit tanaman dari citra daun sudah
ditunjukkan pada skala besar oleh Mohanty et al. (2016).

Kami memilih *object detection*, bukan klasifikasi citra, karena kebutuhan
kasusnya berbeda. Klasifikasi hanya memberi satu label untuk seluruh gambar,
sedangkan deteksi memberi dua informasi sekaligus: jenis penyakit dan lokasi
gejalanya. Informasi lokasi itu penting karena satu helai daun bisa memuat
beberapa bercak sekaligus, dan satu citra lapangan bisa memuat lebih dari
satu kondisi tanaman.

Karena itu, yang ingin kami bangun adalah model deteksi untuk 11 kelas
canonical penyakit dan kondisi tanaman padi dari dataset resmi TELEPATI 8.0,
dengan pipeline yang dapat direproduksi dan diaudit ulang oleh pihak ketiga.

## 2. Rumusan Masalah

Untuk mencapai tujuan itu, kami menurunkannya menjadi enam pertanyaan yang
akan dijawab berurutan sepanjang laporan ini:

1. Bagaimana kondisi aktual dataset resmi yang tersedia, termasuk distribusi
   kelas, karakteristik geometri objek, dan kualitas anotasinya?
2. Masalah kualitas data apa yang perlu ditangani sebelum pemodelan, dan apa
   konsekuensinya bila kami abaikan?
3. Konfigurasi pelatihan seperti apa yang dapat kami pertanggungjawabkan
   berdasarkan eksperimen terkontrol, bukan berdasarkan asumsi?
4. Seberapa baik performa model yang dihasilkan, diukur dengan mAP@0.5 dan
   *F1-score*, dan bagaimana performa itu terdistribusi antar kelas?
5. Pada kondisi seperti apa model kami gagal, dan faktor apa yang konsisten
   dengan kegagalan tersebut?
6. Seberapa jauh seluruh pipeline dapat direproduksi dan diaudit?

Satu batasan perlu kami sampaikan sejak awal karena memengaruhi cara membaca
seluruh angka di laporan ini. Kompetisi melarang penggunaan *external
pretrained weights*, sehingga kami harus melatih model dari inisialisasi
acak. Artinya model tidak mewarisi representasi visual umum dari dataset
besar seperti COCO atau ImageNet, dan itu menurunkan performa yang realistis
dicapai. Kami meminta pembaca menafsirkan setiap hasil berikutnya dalam
konteks batasan tersebut.

## 3. Gambaran Solusi

Dengan enam pertanyaan itu sebagai panduan, kami menyusun pekerjaan sebagai
pipeline bertahap. Setiap tahap menghasilkan artefak yang dapat diperiksa
kembali, sehingga klaim apa pun di laporan ini bisa ditelusuri sampai ke
berkas hasil eksekusinya:

```
Dataset mentah
  -> Audit forensik dataset
  -> Pemetaan 11 kelas canonical
  -> Pemeriksaan kebocoran antar split
  -> Penyiapan data format YOLO
  -> Eksperimen terkontrol (21 percobaan)
  -> Pelatihan model final
  -> Evaluasi dan analisis kesalahan
  -> Audit reproducibility
```

Untuk arsitekturnya kami memakai YOLOv8n, varian terkecil pada keluarga
YOLOv8 (Ultralytics, 2026). Kami memilih varian nano bukan karena keterbatasan
semata, melainkan karena dua kondisi saling menguatkan: model harus dilatih
dari nol tanpa bobot awal, dan perangkat yang tersedia hanya satu laptop
Apple Silicon dengan *backend* MPS. Pada kondisi seperti itu, model
berkapasitas besar yang dilatih dari nol justru lebih sulit dioptimasi dalam
anggaran komputasi yang kami miliki.

Sebelum menyentuh pemodelan, kami mendahulukan pemeriksaan data. Bagian
berikutnya menjelaskan dataset yang kami terima apa adanya.

## 4. Lingkungan Pengembangan dan Reproducibility

Sebelum menyentuh data, kami mencatat lebih dulu identitas lingkungan tempat
seluruh notebook ini dijalankan. Alasannya sederhana: metrik apa pun yang
kami laporkan nanti hanya bermakna bila pembaca tahu pada perangkat dan versi
pustaka seperti apa angka itu lahir. Karena itu kami memperlakukan catatan di
bawah ini sebagai bagian dari jejak audit, bukan sebagai formalitas pembuka.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src" / "agridata").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src" / "agridata").exists(), (
    "Paket agridata tidak ditemukan. Jalankan notebook dari root project atau dari notebooks/."
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Root project: {PROJECT_ROOT}")

In [ ]:
import json
import subprocess
import hashlib
import random
import csv

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import yaml

from agridata.seed import set_global_seed
from agridata.device import detect_device
from agridata.reproducibility.environment import capture_environment_snapshot
from agridata.dataset.mapping import CANONICAL_CLASSES, build_mapping_report
from agridata.dataset.stats import load_canonical_split
from agridata.analysis.dataset_profile import (
    analyze_annotation_overlap,
    audit_missingness,
    compute_bbox_geometry,
    load_duplicate_summary,
    scene_density_summary,
    summarize_class_imbalance,
    summarize_resolution,
)
from agridata.visualization.images import draw_annotated_image
from agridata.visualization.distributions import (
    plot_class_distribution_comparison,
    plot_experiment_overview,
    plot_instances_vs_performance,
    plot_ofat_deltas,
    plot_per_class_ap,
    plot_split_overview,
    plot_threshold_sensitivity,
    plot_training_curves,
)
from agridata.training.train import build_compliant_model, run_training

# agridata.visualization menetapkan backend "Agg" agar dapat dipakai pada
# skrip tanpa layar. Baris berikut mengembalikan backend inline supaya figur
# tampil di dalam notebook.
%matplotlib inline

In [ ]:
env = capture_environment_snapshot()
print(f"Versi Python        : {env['python_version']}")
print(f"Platform            : {env['platform']['platform_string']}")
print(f"Perangkat terpilih  : {env['device']['resolved_device']} (Apple Silicon: {env['device']['is_apple_silicon']})")
print(f"torch               : {env['device']['torch_version']}")
print(f"CUDA tersedia       : {env['device']['cuda_available']}  |  MPS tersedia: {env['device']['mps_available']}")
print(f"Git commit          : {env['git_commit']}")
print(f"Working tree bersih : {env['git_status']['clean']}")

Kami mengunci seluruh sumber keacakan pada satu nilai *seed* yang sama dengan
yang kami pakai pada pelatihan final, mencakup modul `random` pada Python,
NumPy, PyTorch, serta variabel `PYTHONHASHSEED`. Dengan begitu setiap angka
yang muncul di notebook ini berasal dari kondisi acak yang sama setiap kali
dijalankan.


In [ ]:
with open(PROJECT_ROOT / "configs" / "experiment_100epoch_config.yaml") as f:
    FINAL_CONFIG = yaml.safe_load(f)

SEED = FINAL_CONFIG["seed"]
set_global_seed(SEED)

DATASET_ROOT = PROJECT_ROOT / "Telepati 8.0 Datasets"
PREPARED_DIR = PROJECT_ROOT / "data" / "prepared"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"
FIGURES_DIR = PROJECT_ROOT / "artifacts" / "figures" / "final_submission"

assert DATASET_ROOT.exists(), f"Dataset resmi tidak ditemukan pada {DATASET_ROOT}"
print(f"Seed global    : {SEED}")
print(f"Root dataset   : {DATASET_ROOT}")


## 5. Dataset dan Sumber Data

Dengan lingkungan yang sudah terdokumentasi, kami beralih ke bahan dasarnya.
Kami hanya memakai dataset resmi TELEPATI 8.0 dan tidak menambahkan sumber
data lain, sesuai ketentuan yang kami sebutkan di awal.

Dataset sudah terbagi menjadi tiga *split* resmi, yaitu `train`, `valid`, dan
`test`, dan kami mempertahankan pembagian itu apa adanya. Kami sengaja tidak
menggabungkan atau mengacak ulang antar *split*, karena tindakan itu akan
merusak dasar perbandingan dengan peserta lain sekaligus membuka peluang
kebocoran informasi dari data uji ke data latih.

Anotasinya tersimpan dalam format COCO, satu berkas `_annotations.coco.json`
per *split*, yang memuat daftar citra, daftar anotasi *bounding box*, dan
daftar kategori. Struktur inilah yang kami baca pada sel berikutnya.


In [ ]:
raw_counts = {}
for split in ["train", "valid", "test"]:
    with open(DATASET_ROOT / split / "_annotations.coco.json") as f:
        data = json.load(f)
    raw_counts[split] = {
        "citra": len(data["images"]),
        "anotasi": len(data["annotations"]),
        "kategori_mentah": len(data["categories"]),
    }

print(f"{'Split':8s} {'Citra':>8s} {'Anotasi':>10s} {'Kategori mentah':>18s}")
for split, c in raw_counts.items():
    print(f"{split:8s} {c['citra']:8d} {c['anotasi']:10d} {c['kategori_mentah']:18d}")
print(f"\nTotal citra   : {sum(c['citra'] for c in raw_counts.values())}")
print(f"Total anotasi : {sum(c['anotasi'] for c in raw_counts.values())}")

Satu hal langsung menarik perhatian kami dari keluaran di atas. Dataset memuat
21 kategori mentah, sedangkan target deteksi resmi hanya 11 kelas canonical.
Setelah kami periksa, selisih ini bukan kesalahan dataset, melainkan akibat
variasi penulisan label ditambah keberadaan kategori *supercategory* yang
sebenarnya bukan target deteksi. Kami menunda penanganannya sampai Bagian 7,
setelah profiling selesai.


## 6. Eksplorasi dan Profiling Dataset

Mengetahui asal dan struktur dataset belum cukup untuk mengambil keputusan
pemodelan. Kami perlu tahu lebih dulu seperti apa kondisi aktual datanya,
karena karakteristik itulah yang nantinya kami pakai untuk menafsirkan
performa model secara jujur. Tanpa langkah ini, kami hanya akan punya angka
akhir tanpa penjelasan mengapa angkanya seperti itu.

Jadi tujuan bagian ini bukan memajang grafik, melainkan mengumpulkan bukti
tentang tiga hal yang kami duga paling menentukan: seberapa timpang sebaran
kelasnya, seberapa kecil objek yang harus ditemukan model, dan seberapa padat
objek dalam satu citra.

Seluruh perhitungan di bawah memanggil fungsi pada
`src/agridata/analysis/dataset_profile.py` dan dapat kami hasilkan ulang
melalui `python scripts/profile_dataset.py`.


### 6.1 Struktur Dataset

Data dimuat dengan pemetaan canonical sudah diterapkan, sehingga jumlah
anotasi yang ditampilkan adalah jumlah anotasi yang benar-benar menjadi
target deteksi.

In [ ]:
splits = {}
for split in ["train", "valid", "test"]:
    splits[split] = load_canonical_split(DATASET_ROOT, split, "_annotations.coco.json")

split_counts = {
    s: {"images": len(d.images), "annotations": len(d.annotations)} for s, d in splits.items()
}

print(f"{'Split':8s} {'Citra':>8s} {'Anotasi canonical':>20s} {'Rata-rata anotasi/citra':>26s}")
for s, c in split_counts.items():
    rata = c["annotations"] / c["images"]
    print(f"{s:8s} {c['images']:8d} {c['annotations']:20d} {rata:26.2f}")

### 6.2 Distribusi Split

Kami mulai dari yang paling mendasar, yaitu proporsi antar *split*, karena
angka inilah yang menentukan seberapa kuat kesimpulan yang boleh kami tarik
dari evaluasi nanti. *Split* validasi yang terlalu kecil akan membuat metrik
kami bergoyang antar pengulangan, sedangkan *split* latih yang terlalu kecil
membatasi kemampuan model belajar sejak awal.


In [ ]:
fig_path = plot_split_overview(split_counts, FIGURES_DIR / "split_overview.png")
plt.figure(figsize=(9, 5))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

total_img = sum(c["images"] for c in split_counts.values())
for s, c in split_counts.items():
    print(f"{s:8s}: {c['images']/total_img:6.1%} dari total citra")

Proporsinya mendekati pola 76 persen latih, 16 persen validasi, dan 8 persen
uji. Bagi kami yang terpenting adalah ukuran *split* validasi yang melebihi dua
ribu citra, karena jumlah sebesar itu cukup memadai untuk menghasilkan
estimasi metrik yang stabil. Dugaan ini belakangan terbukti pada Bagian 14,
ketika mAP@0.5 tercatat identik pada beberapa kali pengulangan evaluasi.


### 6.3 Distribusi Kelas

Sebelum melihat grafiknya, kami perlu membedakan dua besaran yang mudah
tertukar. Jumlah *instance* adalah banyaknya *bounding box* untuk suatu kelas,
sedangkan jumlah citra adalah banyaknya gambar yang memuat minimal satu
*instance* kelas tersebut.

Keduanya tidak identik karena satu citra dapat memuat banyak *instance* dari
kelas yang sama, dan perbedaan itu ternyata justru menjadi salah satu temuan
paling penting dalam laporan ini, seperti akan terlihat pada Bagian 6.11.


In [ ]:
imbalance = {s: summarize_class_imbalance(d) for s, d in splits.items()}
train_imb = imbalance["train"]

fig_path = plot_class_distribution_comparison(
    train_imb.per_class_instances,
    train_imb.per_class_images,
    "Distribusi kelas pada split train",
    FIGURES_DIR / "train_class_distribution.png",
)
plt.figure(figsize=(10, 6))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

In [ ]:
print(f"{'Kelas':28s} {'Instance':>9s} {'Citra':>7s} {'Instance/citra':>15s} {'Porsi':>8s}")
for cls in CANONICAL_CLASSES:
    n_inst = train_imb.per_class_instances[cls]
    n_img = train_imb.per_class_images[cls]
    rasio = n_inst / n_img if n_img else 0
    print(f"{cls:28s} {n_inst:9d} {n_img:7d} {rasio:15.2f} {train_imb.per_class_instance_share[cls]:7.1%}")

print(f"\nKelas terbanyak : {train_imb.most_frequent_class} ({train_imb.max_instances} instance)")
print(f"Kelas tersedikit: {train_imb.least_frequent_class} ({train_imb.min_instances} instance)")
print(f"Rasio ketidakseimbangan: {train_imb.imbalance_ratio:.1f} kali")

### 6.4 Analisis Ketidakseimbangan Kelas

Grafik di atas sudah memperlihatkan ketimpangan, dan kami ingin mengukurnya.
Rasio antara kelas terbanyak dan kelas tersedikit pada *split* latih mencapai
22,6 kali. Rasio itu ternyata berbeda-beda antar *split*, sehingga kami
memeriksanya satu per satu alih-alih menyimpulkan dari data latih saja.


In [ ]:
print(f"{'Split':8s} {'Terbanyak':<26s} {'Tersedikit':<26s} {'Rasio':>8s}")
for s, imb in imbalance.items():
    print(
        f"{s:8s} {imb.most_frequent_class + ' (' + str(imb.max_instances) + ')':<26s} "
        f"{imb.least_frequent_class + ' (' + str(imb.min_instances) + ')':<26s} {imb.imbalance_ratio:7.1f}x"
    )

Angka di atas berarti model kami menerima frekuensi observasi yang sangat
berbeda antar kelas. Kelas dengan sedikit *instance* memperoleh lebih sedikit
variasi visual selama pelatihan, sehingga kemampuan generalisasinya berpotensi
lebih rendah. Buda et al. (2018) mempelajari pengaruh ketidakseimbangan kelas
pada jaringan konvolusional secara sistematis dan menemukan bahwa
ketidakseimbangan umumnya merugikan performa klasifikasi.

Kami menahan diri untuk tidak berhenti pada kesimpulan sederhana itu. Jumlah
data bukan satu-satunya faktor, dan kami menguji hubungannya secara eksplisit
pada Bagian 17. Yang jelas, temuan ini menjadi konteks yang kami perlukan
ketika nanti membaca perbedaan AP@0.5 antar kelas pada Bagian 15.

Satu hal lagi yang perlu kami catat: rasio ketidakseimbangan pada *split*
validasi dan uji justru lebih tinggi daripada *split* latih, sekitar 32 kali
dan 31 kali. Artinya kami mengevaluasi model pada distribusi yang bahkan
lebih timpang daripada distribusi tempat model itu belajar.

Namun frekuensi kelas saja belum cukup menjelaskan kesulitan deteksi. Dua
kelas dengan jumlah data serupa bisa punya karakteristik objek yang sangat
berbeda. Karena itu pemeriksaan kami lanjutkan dari pertanyaan "berapa banyak
objeknya" menuju "seperti apa ukuran objek yang harus ditemukan model".

### 6.5 Distribusi Ukuran *Bounding Box*

Setelah mengetahui berapa banyak objek per kelas, pertanyaan kami berikutnya
adalah seberapa besar objek yang harus ditemukan model. Ukuran objek merupakan
salah satu faktor paling menentukan dalam *object detection*, dan kami menduga
faktor inilah yang paling menekan hasil kami nanti.

Supaya dapat dibandingkan antar citra dengan resolusi berbeda, kami menghitung
luas *bounding box* relatif terhadap luas citranya sendiri, dan menetapkan
ambang objek kecil pada satu persen luas citra.


In [ ]:
geometry = {s: compute_bbox_geometry(d, small_object_threshold=0.01) for s, d in splits.items()}

plt.figure(figsize=(10, 5))
plt.imshow(mpimg.imread(FIGURES_DIR / "train_bbox_relative_area.png"))
plt.axis("off")
plt.show()

print(f"{'Split':8s} {'Median luas relatif':>22s} {'Median rasio aspek':>21s} {'Porsi objek kecil':>20s}")
for s, g in geometry.items():
    print(f"{s:8s} {g.median_relative_area:22.4%} {g.median_aspect_ratio:21.2f} {g.small_object_share:20.1%}")

Sebarannya sangat condong ke kiri. Pada *split* latih, 38,1 persen *bounding
box* menutupi kurang dari satu persen luas citra, dan proporsinya justru
lebih tinggi lagi pada *split* validasi (43,3 persen) serta uji (46,8 persen).

Temuan ini penting karena model tidak hanya harus mengenali kelas, tetapi
juga menemukan lokasinya. Ketika objek berukuran sangat kecil, pergeseran
beberapa piksel saja sudah mengubah nilai IoU secara berarti, sehingga
lokalisasi menjadi jauh lebih sensitif terhadap resolusi masukan. Feng et al.
(2023) menempatkan deteksi objek kecil sebagai persoalan yang memang diakui
sulit, dengan resolusi fitur masukan sebagai salah satu faktor utamanya.

Justru karena itu kami tidak menetapkan ukuran citra masukan berdasarkan
asumsi. Kami memperlakukannya sebagai parameter yang harus diuji, dan
hasilnya kami laporkan pada Bagian 11.

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(mpimg.imread(FIGURES_DIR / "train_bbox_aspect_ratio.png"))
plt.axis("off")
plt.show()

Distribusi rasio aspeknya terpusat di sekitar nilai satu, yang berarti sebagian
besar *bounding box* mendekati bentuk persegi. Namun kami melihat ekor yang
memanjang ke kanan, yaitu kotak yang jauh lebih lebar daripada tingginya.
Bentuk memanjang seperti itu masuk akal secara domain, karena konsisten dengan
gejala penyakit yang menyebar mengikuti bentuk helai daun.


### 6.6 Resolusi Citra

Variasi resolusi memengaruhi bagaimana citra diubah ukurannya sebelum masuk
ke model, dan karenanya memengaruhi ukuran efektif objek kecil.

In [ ]:
resolution = {s: summarize_resolution(d) for s, d in splits.items()}

plt.figure(figsize=(7, 7))
plt.imshow(mpimg.imread(FIGURES_DIR / "train_image_resolution.png"))
plt.axis("off")
plt.show()

print(f"{'Split':8s} {'Resolusi unik':>15s} {'Resolusi dominan':>20s} {'Porsi dominan':>16s}")
for s, r in resolution.items():
    dom = f"{r.most_common_resolution[0]}x{r.most_common_resolution[1]}"
    print(f"{s:8s} {r.distinct_resolutions:15d} {dom:>20s} {r.most_common_share:15.1%}")

### 6.7 *Missingness* dan Validitas Anotasi

Kami tidak bisa memindahkan begitu saja istilah *missing value* dari data
tabular ke dataset deteksi objek. Pada konteks ini kami mengartikan
*missingness* sebagai relasi yang putus antara berkas JSON dan berkas citra,
atau parameter *bounding box* yang tidak dapat mendeskripsikan suatu wilayah.

Kami melaporkan setiap pemeriksaan meskipun hasilnya nol, karena nilai nol
merupakan hasil audit yang sah dan bukan tanda bahwa pemeriksaannya tidak kami
lakukan.


In [ ]:
missing = {s: audit_missingness(DATASET_ROOT, s, "_annotations.coco.json") for s in splits}

for s, report in missing.items():
    print(f"--- split {s} ---")
    for c in report.checks:
        status = "OK" if c.count == 0 else "PERLU DITINJAU"
        print(f"  {c.count:6d} / {c.total:6d} ({c.percentage:5.2f}%)  {c.name:58s} {status}")
    print()

Hasilnya bersih pada sisi integritas referensi. Kami tidak menemukan berkas
citra yang hilang, berkas tanpa *record* JSON, anotasi yang merujuk citra atau
kategori yang tidak ada, *bounding box* kosong maupun berdimensi tidak valid,
maupun metadata dimensi citra yang hilang.

Dua pemeriksaan menghasilkan nilai bukan nol, dan keduanya perlu kami jelaskan
agar tidak terbaca sebagai cacat.

Yang pertama adalah citra tanpa anotasi, sebanyak 59 pada *train*, 13 pada
*valid*, dan 5 pada *test*. Citra semacam ini tidak memuat objek target, tetapi
pada kerangka *object detection* ia tetap berguna sebagai contoh latar
belakang, sehingga kami tidak memperlakukannya sebagai kesalahan. Yang kami
catat hanyalah bahwa jumlah citra efektif yang memuat target deteksi sedikit
lebih rendah daripada jumlah citra total.

Yang kedua adalah kategori tanpa anotasi, tiga kategori pada setiap *split*,
yaitu `Leaf-blight`, `Rice-Leaf-Diseasee`, dan `paddy`. Ketiganya merupakan
*supercategory* dan bukan target deteksi, sehingga temuan ini justru menguatkan
keputusan pemetaan yang kami ambil di Bagian 7.


### 6.8 *Duplicate* dan *Data Leakage*

Pemeriksaan berikut ini yang paling kami khawatirkan hasilnya. Kesamaan citra
antar *split* merupakan risiko serius, karena model bisa saja kami evaluasi
pada citra yang sudah pernah dilihatnya saat pelatihan, dan skor yang dihasilkan
akan menyesatkan tanpa kami sadari. Audit forensik pada tahap awal project
memeriksa hal ini menggunakan *hash* konten berkas.


In [ ]:
duplicates = load_duplicate_summary(PROJECT_ROOT / "artifacts" / "audit" / "dataset_audit_report.json")

if duplicates["available"]:
    for pair, n in duplicates["pairs"].items():
        print(f"{pair:18s}: {n} citra duplikat persis")
    print(f"\nTotal duplikat persis lintas split: {duplicates['total_exact_duplicates']}")
    for pair, matches in duplicates["matches"].items():
        for m in matches:
            print(f"\n  pasangan pada {pair}:")
            for k, v in m.items():
                print(f"    {k}: {v}")
else:
    print(duplicates["reason"])

Kekhawatiran kami terbukti, meski dalam skala kecil. Kami menemukan satu
pasangan citra dengan konten identik antara *split* latih dan *split* uji.

Penanganannya kami buat sespesifik mungkin supaya dapat diaudit. Dataset mentah
tidak kami ubah sama sekali. Pada *manifest* data siap latih, entri pada sisi
`train` kami keluarkan. Sementara *split* validasi dan uji tidak kami modifikasi
sedikit pun.

Dengan cara itu risiko kebocoran informasi berkurang tanpa kami mengubah dasar
evaluasi resmi. Pemeriksaan tambahan berbasis *perceptual hash* memang masih
menyisakan sejumlah kandidat kemiripan yang belum kami verifikasi satu per satu
secara visual, dan keterbatasan itu kami catat pada Bagian 18.


### 6.9 Anotasi yang Saling Bertumpuk

Selain duplikat antar *split*, kami juga memeriksa kualitas anotasi di dalam
satu citra. Dua kotak pada kelas yang sama dengan tumpang tindih tinggi dapat
menandakan objek yang sama dianotasi dua kali.

Tumpang tindih antar kelas berbeda kami perlakukan terpisah, karena pada citra
daun hal itu sering kali wajar, misalnya ketika dua gejala berbeda muncul pada
helai yang sama.


In [ ]:
overlap = {s: analyze_annotation_overlap(d, iou_threshold=0.5) for s, d in splits.items()}

print(f"{'Split':8s} {'Pasangan dicek':>16s} {'Kelas sama':>12s} {'Antar kelas':>13s} {'Citra terdampak':>17s}")
for s, o in overlap.items():
    print(
        f"{s:8s} {o['pairs_checked']:16d} {o['overlapping_same_class']:12d} "
        f"{o['overlapping_cross_class']:13d} {o['images_affected']:9d} ({o['share_images_affected']:5.1%})"
    )

print("\nContoh tumpang tindih kelas sama pada split train:")
for ex in overlap["train"]["same_class_examples"][:5]:
    print(f"  image_id={ex['image_id']:6d}  {ex['kelas']:24s}  IoU={ex['iou']:.4f}")

Jumlahnya kecil, yaitu 85 pasang pada *train*, 16 pada *valid*, dan 13 pada
*test*, yang secara keseluruhan menyentuh kurang dari satu persen citra. Hampir
seluruhnya terjadi pada kelas yang sama, sedangkan tumpang tindih antar kelas
hanya kami temukan satu kejadian pada seluruh dataset.

Satu contoh bahkan mencapai IoU 0,9519, yang secara praktis berarti dua kotak
menandai objek yang nyaris identik. Kasus seperti itu paling mungkin merupakan
anotasi ganda.

Meski begitu, kami tidak menghapus atau mengubah satu pun anotasi berdasarkan
temuan ini. Jumlahnya terlalu kecil untuk memengaruhi hasil secara berarti, dan
menurut kami mengubah anotasi resmi berdasarkan dugaan otomatis justru berisiko
lebih besar daripada manfaatnya. Kami mencatatnya sebagai karakteristik data,
bukan sebagai cacat yang perlu diperbaiki.


### 6.10 Contoh Visual Dataset

Sampel diambil secara deterministik menggunakan *seed* global, sehingga
citra yang sama akan muncul pada setiap eksekusi.

In [ ]:
train_data = splits["train"]
ann_by_image = {}
for ann in train_data.annotations:
    ann_by_image.setdefault(ann.image_id, []).append(ann)

set_global_seed(SEED)
sample_ids = random.sample(sorted(ann_by_image.keys()), 4)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, image_id in zip(axes, sample_ids):
    rec = train_data.images_by_id[image_id]
    annotated = draw_annotated_image(DATASET_ROOT / "train" / rec.file_name, rec, ann_by_image[image_id])
    ax.imshow(annotated)
    ax.set_title(f"{len(ann_by_image[image_id])} anotasi", fontsize=10)
    ax.axis("off")
plt.suptitle("Contoh citra latih beserta anotasi ground truth", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
density = {s: scene_density_summary(d) for s, d in splits.items()}
print(f"{'Split':8s} {'Median anotasi/citra':>22s} {'Maksimum':>10s} {'Citra padat (>3)':>18s}")
for s, dd in density.items():
    print(f"{s:8s} {dd['median_annotations_per_image']:22.0f} {dd['max_annotations_in_one_image']:10d} {dd['crowded_share']:17.1%}")

### 6.11 Kasus Khas: Adegan Padat, Objek Kecil, dan Kasus Sulit

Angka-angka di atas memberi gambaran agregat, tetapi kami ingin melihat sendiri
seperti apa kasus tersulit dalam dataset ini. Supaya pemilihannya tidak bisa
dituduh menguntungkan kami, aturan seleksinya kami tetapkan lebih dulu dan kami
cetak bersama laporannya, bukan dipilih berdasarkan penampilan.

Untuk adegan padat kami mengambil citra dengan jumlah anotasi terbanyak. Untuk
objek kecil kami mengambil citra dengan proporsi kotak di bawah satu persen luas
citra tertinggi, dibatasi pada citra yang memuat minimal tiga anotasi. Dan untuk
kasus sulit kami memakai hasil kali jumlah anotasi dengan proporsi objek kecil.


In [ ]:
cases_report_path = REPORTS_DIR / "dataset_cases_train.json"
if not cases_report_path.exists():
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "visualize_dataset_cases.py"),
         "--dataset-root", str(DATASET_ROOT), "--split", "train"],
        check=True, cwd=PROJECT_ROOT,
    )

with open(cases_report_path) as f:
    cases_report = json.load(f)

for key, judul in (("crowded", "Adegan padat"), ("small_objects", "Objek kecil"), ("difficult", "Kasus sulit")):
    path = FIGURES_DIR / f"{key}_train.png"
    if path.exists():
        plt.figure(figsize=(17, 6))
        plt.imshow(mpimg.imread(path))
        plt.axis("off")
        plt.show()

d_train = density["train"]
print(f"Anotasi terbanyak dalam satu citra : {d_train['max_annotations_in_one_image']}")
print(f"Median anotasi per citra           : {d_train['median_annotations_per_image']:.0f}")

Ketiga citra terpadat ternyata seluruhnya merupakan kelas Brown spot, dengan
178, 126, dan 90 anotasi dalam satu gambar. Bagi kami inilah penjelasan visual
paling gamblang mengapa Brown spot memiliki jumlah *instance* terbanyak (5.010)
sekaligus AP@0.5 terendah: gejalanya berupa puluhan bercak kecil yang berdesakan
pada satu helai daun.

Temuan ini menyambungkan beberapa hal yang sebelumnya kami catat terpisah.
Brown spot punya rasio *instance* per citra tertinggi, objek kecil mendominasi
dataset seperti yang kami lihat pada Bagian 6.5, adegan padat memiliki rasio
*false negative* jauh lebih tinggi yang akan kami tunjukkan pada Bagian 16, dan
Brown spot memperoleh AP@0.5 terendah pada Bagian 15.

Rantai penjelasan itu konsisten satu sama lain, tetapi kami perlu menegaskan
bahwa sifatnya observasional. Kami tidak menjalankan eksperimen terkontrol untuk
membuktikan bahwa kepadatan adegan yang menyebabkan AP rendah.


### Ringkasan Bagian 6

Dari seluruh pemeriksaan di atas, ada lima hal yang kami bawa ke tahap
pemodelan.

Dataset menyediakan 13.298 citra dengan 27.721 anotasi canonical, jumlah yang
menurut kami memadai untuk melatih model deteksi berukuran kecil. Namun
distribusi kelasnya sangat timpang, dengan rasio 22,6 kali pada *split* latih
dan lebih dari 30 kali pada *split* validasi serta uji. Objek berukuran kecil
juga mendominasi, yaitu 38,1 persen pada *split* latih dan 46,8 persen pada
*split* uji.

Di sisi kualitas data, integritas referensi anotasi bersih pada seluruh
pemeriksaan, dengan dua catatan yang sudah kami jelaskan yaitu citra tanpa
anotasi dan *supercategory* tanpa anotasi. Satu duplikat persis lintas *split*
kami temukan dan kami tangani pada tingkat *manifest*, tanpa mengubah dataset
mentah.

Kelima karakteristik inilah yang nanti kami pakai untuk membaca hasil model.
Tetapi sebelum sampai ke pemodelan, ada satu pekerjaan yang harus kami
selesaikan lebih dulu: menyatukan label yang tidak konsisten.


## 7. Pemetaan 11 Kelas Canonical

Salah satu temuan profiling di atas menuntut tindakan sebelum pelatihan bisa
dimulai. Dataset mentah memuat 21 kategori, sedangkan target deteksi resmi
hanya 11 kelas. Setelah kami telusuri, selisih itu berasal dari tiga sumber:
variasi penulisan nama untuk konsep yang sama, variasi kapitalisasi dan tanda
hubung, serta kategori *supercategory* yang sebenarnya bukan target deteksi.

Kami tidak bisa membiarkannya. Tanpa penyatuan ini, label yang secara semantik
sama akan diperlakukan sebagai kelas yang berbeda, sehingga data untuk satu
penyakit terpecah ke beberapa kelas dan model kami paksa memisahkan sesuatu
yang sebenarnya identik. Bagian ini memaparkan pemetaan yang kami susun
beserta pemeriksaan bahwa tidak ada satu pun kategori yang tertinggal.


In [ ]:
with open(DATASET_ROOT / "train" / "_annotations.coco.json") as f:
    train_raw = json.load(f)

mapping_report = build_mapping_report(train_raw["categories"])

print(f"Kategori mentah          : {mapping_report['total_raw_categories']}")
print(f"Terpetakan ke canonical  : {len(mapping_report['mapped'])}")
print(f"Supercategory dikecualikan: {[p['raw_name'] for p in mapping_report['supercategory_placeholders']]}")
print(f"Tidak terpetakan         : {mapping_report['unmapped_raw_categories']} (harus kosong)")
assert not mapping_report["unmapped_raw_categories"], "Ada kategori mentah yang belum terpetakan."

print(f"\n{'Label mentah':32s} -> {'Kelas canonical':28s} {'ID model'}")
for m in sorted(mapping_report["mapped"], key=lambda r: r["canonical_id"]):
    print(f"{m['raw_name']:32s} -> {m['canonical_name']:28s} {m['canonical_id'] - 1}")

Kami membuat pemetaan ini eksplisit dan sengaja gagal secara keras bila
menemukan kategori mentah yang tidak dikenali, supaya perubahan dataset di masa
depan tidak lolos diam-diam tanpa kami sadari. Versi tabel pemetaannya kami
catat sebagai `MAPPING_VERSION` agar setiap perubahan dapat dilacak.

Tiga kategori yang kami kecualikan bukan kami hapus sewenang-wenang. Ketiganya
terbukti tidak memiliki satu pun anotasi pada seluruh *split*, sebagaimana
sudah terlihat pada audit *missingness* di Bagian 6.7.


## 8. Persiapan Data

Setelah pemetaan kami kunci, langkah berikutnya adalah mengubah dataset mentah
menjadi data siap latih dalam format YOLO. Kami menjalankannya sebagai satu
alur tetap:

```
Dataset mentah
  -> Validasi struktur dan anotasi
  -> Pemetaan canonical
  -> Pemeriksaan kebocoran antar split
  -> Penulisan manifest dan label format YOLO
```

Ada empat prinsip yang kami pegang sepanjang tahap ini, dan ketiganya yang
pertama kami pilih demi auditabilitas. Dataset mentah tidak pernah kami ubah,
pindah, atau timpa, sehingga juri selalu bisa kembali ke kondisi awal. Citra
tidak kami salin melainkan kami rujuk lewat *symlink*, agar tidak ada
duplikasi byte yang membengkakkan repository. Hasil penyiapan dapat kami
hasilkan ulang dari *seed* yang sama dan menghasilkan *manifest* yang identik
byte per byte. Terakhir, pengecualian akibat duplikat hanya kami terapkan pada
sisi `train`, supaya `valid` dan `test` tetap utuh sebagaimana panitia
membagikannya.


In [ ]:
if not (PREPARED_DIR / "data.yaml").exists():
    print("Menyiapkan dataset siap latih...")
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "prepare_dataset.py"),
         "--dataset-root", str(DATASET_ROOT), "--output-dir", str(PREPARED_DIR), "--seed", str(SEED)],
        check=True, cwd=PROJECT_ROOT,
    )
else:
    print(f"Data siap latih sudah tersedia pada {PREPARED_DIR}")

with open(REPORTS_DIR / "dataset_preparation_summary.json") as f:
    prep = json.load(f)

print(f"\nSeed             : {prep['seed']}")
print(f"Versi pemetaan   : {prep['mapping_version']}")
print(f"\n{'Split':8s} {'Citra disiapkan':>17s} {'Dikecualikan (leakage)':>24s} {'Anotasi':>10s}")
for s in prep["splits"]:
    print(f"{s['split']:8s} {s['num_images_prepared']:17d} {s['num_images_excluded_leakage']:24d} {s['num_annotations_prepared']:10d}")

### Ringkasan Bagian 7 dan 8

Sampai di titik ini kami sudah menyatukan 21 kategori mentah menjadi 11 kelas
canonical tanpa menyisakan kategori yang tidak terpetakan, dan mengecualikan
tiga *supercategory* berdasarkan bukti bahwa ketiganya memang tidak memuat
anotasi sama sekali. Satu citra kami keluarkan dari *manifest* latih akibat
duplikat lintas *split*, sementara dataset mentahnya tetap utuh. Seluruh proses
penyiapan bersifat deterministik dan dapat diverifikasi ulang.

Dengan data yang sudah siap dan terdokumentasi, kami beralih ke pertanyaan
berikutnya, yaitu bagaimana model sebaiknya dilatih.


## 9. Strategi Eksperimen

Dengan data siap latih di tangan, pertanyaannya berpindah: konfigurasi seperti
apa yang sebaiknya kami pakai? Kami memutuskan untuk tidak menjawabnya dengan
nilai *default* maupun intuisi. Sebagai gantinya kami menguji satu faktor pada
satu waktu, yaitu mengubah satu parameter sambil menahan parameter lain tetap,
supaya setiap perubahan hasil benar-benar dapat kami atribusikan pada faktor
yang kami ubah.

Anggaran komputasi yang kami punya memaksa satu kompromi. Pengujian kami
jalankan pada skala penyaringan, yaitu memakai sebagian data latih dan jumlah
*epoch* yang kecil. Kami mencatat konsekuensinya sebagai keterbatasan pada
Bagian 18, karena hasil pada skala penyaringan tidak dijamin berlaku sama pada
skala penuh, dan kami tidak ingin pembaca menyimpulkan lebih jauh dari yang
buktinya sanggup dukung.

Enam pertanyaan berikut yang kami bawa ke meja eksperimen:

1. Apakah ukuran citra masukan memengaruhi performa deteksi?
2. Apakah jumlah *epoch* masih memberikan perbaikan pada rentang yang diuji?
3. Apakah *optimizer* tertentu lebih stabil daripada yang lain?
4. Apakah *learning rate* yang lebih kecil membantu?
5. Apakah *augmentation* memberikan manfaat pada skala data ini?
6. Apakah penyeimbangan kelas melalui *oversampling* memperbaiki kelas minoritas?


## 10. Baseline dan Eksperimen Terkontrol

Setiap percobaan kami catat pada `artifacts/experiments/experiment_log.json`
beserta *seed*, *hyperparameter*, arsitektur, *hash manifest* dataset, dan
*commit* Git pada saat percobaan dijalankan, sehingga siapa pun dapat menelusuri
satu baris angka kembali ke kondisi persis yang menghasilkannya.

Sebelum membaca angkanya, kami perlu menitipkan satu peringatan. Seluruh
percobaan di bagian ini berjalan pada skala penyaringan, yaitu sebagian kecil
data latih dengan *epoch* yang sangat sedikit, sehingga nilai mAP@0.5 yang
muncul berada pada kisaran ribuan per seribu dan tidak sebanding dengan hasil
model final pada Bagian 14. Kami memakai angka ini semata untuk membandingkan
faktor satu sama lain, bukan untuk menyatakan performa.

Kami juga sengaja menyembunyikan kolom *F1* dari tabel di bawah. Kolom itu
bernilai nol untuk seluruh percobaan karena metriknya memang tidak kami hitung
pada tahap penyaringan, dan menampilkannya hanya akan mengundang salah tafsir
seolah model gagal total.


In [ ]:
with open(PROJECT_ROOT / "artifacts" / "experiments" / "experiment_log.json") as f:
    experiments = json.load(f)

print(f"Jumlah percobaan tercatat: {len(experiments)}\n")
print(f"{'ID':4s} {'imgsz':>6s} {'batch':>6s} {'epoch':>6s} {'optim':>7s} {'lr':>9s} {'mAP@0.5':>9s} {'Durasi(s)':>10s}")
for e in experiments:
    print(
        f"{e['experiment_id']:4s} {e['image_size']:6d} {e['batch_size']:6d} {e['epochs']:6d} "
        f"{e['optimizer']:>7s} {e['learning_rate']:9.5f} {e['best_val_map50']:9.4f} "
        f"{e['training_duration_seconds']:10.0f}"
    )

In [ ]:
fig_path = plot_experiment_overview(experiments, FIGURES_DIR / "experiment_overview.png")
plt.figure(figsize=(12, 5))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

### 10.1 Titik Awal

Percobaan pertama kami, E01, sebenarnya hanya uji asap: dua *epoch* pada lima
persen data latih. Tujuannya bukan mengukur performa melainkan membuktikan
bahwa pipeline kami berjalan dari ujung ke ujung. Hasilnya nol, persis seperti
yang kami harapkan pada anggaran sekecil itu.

Titik referensi yang sesungguhnya adalah E02, yang kami jadikan *baseline* untuk
seluruh perbandingan satu faktor. Setiap varian sesudahnya mengubah tepat satu
parameter terhadap E02, sementara parameter lain kami tahan tetap.


### 10.2 Pengujian Satu Faktor pada Satu Waktu

Tabel berikut menghitung selisih tiap varian terhadap *baseline* E02 secara
langsung dari log percobaan.

In [ ]:
by_id = {e["experiment_id"]: e for e in experiments}
BASELINE_ID = "E02"
baseline_map = by_id[BASELINE_ID]["best_val_map50"]

ofat_factors = {
    "E03": "ukuran citra 320 -> 640",
    "E04": "batch 16 -> 32",
    "E05": "learning rate 0,001 -> 0,0001",
    "E06": "optimizer AdamW -> SGD",
    "E07": "kekuatan augmentasi diubah",
    "E08": "epoch 5 -> 10",
}

ofat_rows = []
for exp_id, factor in ofat_factors.items():
    value = by_id[exp_id]["best_val_map50"]
    ofat_rows.append({
        "experiment_id": exp_id,
        "factor": factor,
        "map50": value,
        "delta": value - baseline_map,
    })

print(f"Baseline {BASELINE_ID}: mAP@0.5 = {baseline_map:.4f}\n")
print(f"{'ID':5s} {'Faktor yang diubah':34s} {'mAP@0.5':>9s} {'Selisih':>10s} {'Arah'}")
for r in sorted(ofat_rows, key=lambda r: r["delta"], reverse=True):
    arah = "membantu" if r["delta"] > 0 else "menurunkan"
    print(f"{r['experiment_id']:5s} {r['factor']:34s} {r['map50']:9.4f} {r['delta']:+10.4f} {arah}")

In [ ]:
fig_path = plot_ofat_deltas(BASELINE_ID, ofat_rows, FIGURES_DIR / "ofat_deltas.png")
plt.figure(figsize=(10, 5))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

Dari enam pertanyaan yang kami ajukan, hanya dua faktor yang terbukti
memperbaiki hasil, sementara empat sisanya justru menurunkannya.

Pengaruh terbesar datang dari durasi pelatihan. Menaikkan *epoch* dari 5
menjadi 10 menaikkan mAP@0.5 sekitar tujuh kali lipat terhadap *baseline*,
yang kami baca sebagai tanda bahwa model masih jauh dari jenuh pada anggaran
tersebut. Untuk model yang dilatih dari nol tanpa bobot awal, gejala seperti
ini memang wajar kami temui.

Faktor kedua yang membantu adalah ukuran citra: resolusi 640 lebih baik
daripada 320. Perbaikan ini tidak berdiri sendiri, melainkan konsisten dengan
temuan kami pada Bagian 6.5 bahwa objek kecil mendominasi dataset, dan
belakangan diperkuat lagi oleh analisis kesalahan pada Bagian 16.

Sebaliknya, batch yang lebih besar, *learning rate* yang lebih kecil,
*optimizer* SGD, dan perubahan kekuatan augmentasi seluruhnya menurunkan hasil
pada anggaran ini. Karena dua temuan pertama saling mendukung, kami
menggabungkan keduanya menjadi percobaan konfirmasi pada Bagian 10.5.


### 10.3 Ablasi Augmentasi

Block 11 menguji setiap jenis augmentasi secara terpisah terhadap referensi
tanpa augmentasi sama sekali.

In [ ]:
aug_ids = ["E09", "E10", "E11", "E12", "E13", "E14", "E15", "E16", "E17"]
aug_labels = {
    "E09": "tanpa augmentasi (referensi)",
    "E10": "horizontal flip",
    "E11": "vertical flip",
    "E12": "rotasi",
    "E13": "scaling",
    "E14": "translasi",
    "E15": "brightness dan contrast",
    "E16": "color transform",
    "E17": "mosaic",
}
aug_ref = by_id["E09"]["best_val_map50"]

print(f"Referensi E09 tanpa augmentasi: mAP@0.5 = {aug_ref:.4f}\n")
print(f"{'ID':5s} {'Augmentasi':30s} {'mAP@0.5':>9s} {'Selisih':>10s}")
for exp_id in aug_ids[1:]:
    v = by_id[exp_id]["best_val_map50"]
    print(f"{exp_id:5s} {aug_labels[exp_id]:30s} {v:9.4f} {v - aug_ref:+10.4f}")

n_worse = sum(1 for e in aug_ids[1:] if by_id[e]["best_val_map50"] < aug_ref)
print(f"\nJumlah augmentasi yang berada di bawah referensi: {n_worse} dari {len(aug_ids) - 1}")

Hasilnya perlu kami sampaikan apa adanya, meski tidak sesuai harapan: seluruh
augmentasi yang kami uji secara individual berada di bawah referensi tanpa
augmentasi pada anggaran lima *epoch*.

Kami tidak menganggap ini aneh. Pola seperti itu umum dijumpai pada anggaran
pelatihan yang sangat pendek, karena augmentasi memperbesar variasi data
sehingga model membutuhkan lebih banyak iterasi untuk mencapai tingkat yang
sama, dan pada anggaran pendek efek tersebut terbaca sebagai penurunan.

Karena itu kami tidak langsung menjadikan hasil tahap ini sebagai dasar
keputusan. Kami mengujinya ulang pada anggaran yang lebih besar, dan hasilnya
kami bahas pada Bagian 10.5.


### 10.4 Ablasi Ketidakseimbangan Kelas

Ketimpangan kelas yang kami temukan pada Bagian 6.4 menuntut jawaban, jadi kami
menguji apakah *oversampling* kelas minoritas memperbaiki hasil.

Satu hal penting kami jaga di sini: *oversampling* hanya kami terapkan pada
*split* latih, sedangkan *split* validasi dan uji tetap menunjuk ke berkas asli
yang identik. Tanpa penjagaan itu, perbandingannya tidak akan sah.


In [ ]:
imb_ref = by_id["E18"]["best_val_map50"]
imb_over = by_id["E19"]["best_val_map50"]
print(f"E18 distribusi kelas alami      : mAP@0.5 = {imb_ref:.4f}")
print(f"E19 oversampling kelas minoritas: mAP@0.5 = {imb_over:.4f}")
print(f"Selisih                         : {imb_over - imb_ref:+.4f}")

Jawabannya negatif. *Oversampling* menurunkan hasil secara jelas pada skala
penyaringan, sehingga kami tidak mengadopsinya pada konfigurasi final.

Konsekuensinya kami terima secara sadar: ketidakseimbangan kelas tetap ada pada
model final kami, dan dampaknya akan terlihat pada sebaran AP antar kelas di
Bagian 15.


### 10.5 Percobaan Konfirmasi

Dua percobaan terakhir kami susun untuk menguji apakah temuan satu faktor tadi
masih bertahan ketika digabungkan. Kami menyatukan dua faktor yang terbukti
paling berpengaruh, yaitu ukuran citra 640 dan jumlah *epoch* yang lebih besar,
lalu membandingkan kondisi dengan dan tanpa augmentasi pada anggaran yang sama.


In [ ]:
e20, e21 = by_id["E20"], by_id["E21"]
print(f"{'ID':5s} {'Kondisi':34s} {'imgsz':>6s} {'epoch':>6s} {'mAP@0.5':>9s}")
print(f"{'E20':5s} {'augmentasi default':34s} {e20['image_size']:6d} {e20['epochs']:6d} {e20['best_val_map50']:9.4f}")
print(f"{'E21':5s} {'tanpa augmentasi':34s} {e21['image_size']:6d} {e21['epochs']:6d} {e21['best_val_map50']:9.4f}")
print(f"\nSelisih E21 terhadap E20: {e21['best_val_map50'] - e20['best_val_map50']:+.4f}")
print(f"Kedua percobaan ini merupakan dua hasil tertinggi dari seluruh {len(experiments)} percobaan.")

Penggabungan resolusi 640 dengan 12 *epoch* menghasilkan dua nilai tertinggi
dari seluruh percobaan kami, yang mengonfirmasi temuan satu faktor sebelumnya.

Pada perbandingan augmentasi, jaraknya berubah menarik. Selisih antara kondisi
tanpa augmentasi dan dengan augmentasi menyempit drastis dibandingkan hasil lima
*epoch* sebelumnya, meskipun kondisi tanpa augmentasi masih sedikit unggul.
Arah tren inilah yang nanti kami pakai sebagai pertimbangan pada Bagian 12.


## 11. Konfigurasi Final

Hasil eksperimen di atas kami tuangkan menjadi satu konfigurasi yang kemudian
kami bekukan pada `configs/final_model_config.yaml`. Setiap parameter kami
sertai alasan yang merujuk ke percobaan tertentu, supaya juri dapat menelusuri
kembali dari mana angka itu datang, bukan menerimanya sebagai pilihan yang
tiba-tiba muncul.


In [ ]:
for k, v in FINAL_CONFIG.items():
    print(f"{k:18s}: {v}")

## 12. Pemilihan Konfigurasi Final

Kami memisahkan penjelasan ini menjadi tiga bagian, karena tidak semua
parameter kami pilih dengan dasar yang sama kuat, dan kami merasa perlu jujur
soal itu.

### Parameter yang mengikuti bukti eksperimen secara langsung

Ukuran citra kami tetapkan 640 karena E03 mengungguli *baseline* E02 sebagai
faktor tunggal, sementara E20 dan E21 menempati dua posisi teratas. Analisis
kesalahan pada Bagian 16 kemudian memperkuatnya dengan menunjukkan bahwa objek
yang terlewat cenderung berukuran kecil.

Batch kami tahan di 16 karena E04 dengan batch 32 justru berada di bawah
*baseline*, yang konsisten dengan berkurangnya jumlah pembaruan gradien per
*epoch*.

Untuk *optimizer* kami memilih AdamW dengan *learning rate* 0,001, sebab E06
dengan SGD maupun E05 dengan *learning rate* 0,0001 sama-sama berada di bawah
*baseline*. AdamW sendiri merupakan varian Adam dengan *weight decay* yang
dipisahkan dari langkah gradien (Loshchilov & Hutter, 2019).

Distribusi kelas kami biarkan alami tanpa *oversampling*, karena E19 menurunkan
hasil secara jelas dibandingkan E18. Dan anggaran *epoch* kami perbesar sampai
50, mengikuti E08 yang menempatkan durasi pelatihan sebagai faktor paling
berpengaruh.

### Satu keputusan yang tidak mengikuti selisih metrik

Kami tetap mengaktifkan augmentasi *default* meskipun E21 tanpa augmentasi
sedikit mengungguli E20 dengan augmentasi. Kami menyatakan ini terbuka sebagai
penilaian kami, bukan sebagai kesimpulan yang ditarik dari data.

Pertama, selisihnya sangat kecil dan terus mengecil seiring bertambahnya
anggaran pelatihan, dari perbedaan yang cukup terasa pada lima *epoch* menjadi
tipis pada 12 *epoch*. Arah tren itu konsisten dengan manfaat augmentasi yang
baru muncul pada pelatihan panjang, sedangkan model final kami latih 50
*epoch*, jauh di atas anggaran pengujian.

Kedua, model ini pada akhirnya ditujukan untuk kondisi lapangan yang bervariasi
dari sisi pencahayaan, sudut, dan jarak pengambilan. Dataset ini sendiri tidak
menjamin cakupan seluruh variasi tersebut, sedangkan augmentasi citra merupakan
pendekatan yang lazim untuk memperluas keragaman data latih dan mengurangi
*overfitting* (Shorten & Khoshgoftaar, 2019).

Kami mencatat keputusan ini sebagai keterbatasan, karena kami tidak punya
percobaan pada 50 *epoch* yang membandingkan kedua kondisi secara langsung.
Pilihan ini berdiri di atas penalaran domain dan arah tren, bukan di atas bukti
langsung pada skala penuh.

Satu pengecualian kami terapkan pada augmentasi, yaitu menonaktifkan pembalikan
vertikal. Tanaman padi tumbuh mengikuti arah gravitasi, sehingga citra yang
dibalik vertikal tidak merepresentasikan kondisi yang akan ditemui model saat
*deployment*.

### Perubahan jumlah *epoch* setelah hasil pertama

Konfigurasi kami awalnya menetapkan 20 *epoch* karena pertimbangan tenggat, dan
pelatihan itu menghasilkan mAP@0.5 sebesar 0,5620. Kami kemudian memperpanjang
anggaran menjadi 50 *epoch* dan memperoleh 0,6277.

Perlu kami nyatakan terbuka bahwa perpanjangan ini kami putuskan setelah hasil
20 *epoch* diketahui. Model 50 *epoch* merupakan pelatihan baru dari nol, bukan
kelanjutan dari *checkpoint* sebelumnya, karena *checkpoint* tersebut sudah
dilucuti keadaan *optimizer*-nya. Seluruh hasil 20 *epoch* tetap kami arsipkan
pada `artifacts/archive/20epoch_run/` dan tidak kami hapus.


## 13. Pelatihan Model Final

Dengan konfigurasi yang sudah beku, kami menjalankan pelatihan model final.
Model kami bangun dari definisi arsitektur `yolov8n.yaml` dengan
`pretrained=False`.

Kami tidak ingin kepatuhan ini bergantung pada kedisiplinan kami sendiri saat
mengetik perintah, jadi kami memasangnya di dalam kode. Fungsi
`build_compliant_model` menolak berjalan bila diberi `pretrained=True` atau
bila argumen arsitekturnya menyerupai berkas *checkpoint*, sehingga pelanggaran
aturan kompetisi akan gagal secara keras, bukan lolos diam-diam.


In [ ]:
compliant_model = build_compliant_model(FINAL_CONFIG["model_arch"], pretrained=False)
print(f"Model dibangun dari definisi arsitektur '{FINAL_CONFIG['model_arch']}'. Task: {compliant_model.task}")
print("Tidak ada checkpoint eksternal yang dirujuk maupun diunduh (YOLO_OFFLINE aktif).")

### Mode eksekusi

Kami menyadari bahwa juri mungkin tidak punya waktu untuk menunggu pelatihan
berjam-jam hanya untuk membaca notebook ini, jadi kami menyediakan dua mode.

Mode evaluasi (`SKIP_TRAINING = True`, dan ini yang kami jadikan *default*)
memuat *final weights* yang sudah dilatih, sehingga notebook dapat dibaca dan
diverifikasi seketika. Mode reproduksi penuh (`SKIP_TRAINING = False`)
menjalankan ulang pelatihan dari konfigurasi beku, dan memerlukan sekitar 7,3
jam pada perangkat Apple Silicon dengan *backend* MPS.

Perlu kami tegaskan bahwa kode pelatihan pada mode kedua bukan tiruan atau
versi sederhana, melainkan fungsi yang persis sama dengan yang menghasilkan
*weights* final kami.


In [ ]:
SKIP_TRAINING = True

FINAL_WEIGHTS_PATH = PROJECT_ROOT / "runs" / "detect" / "final" / "model_100epoch" / "weights" / "best.pt"

if SKIP_TRAINING:
    assert FINAL_WEIGHTS_PATH.exists(), (
        f"SKIP_TRAINING=True tetapi weights final tidak ditemukan pada {FINAL_WEIGHTS_PATH}.\n"
        "Berkas bobot tidak dikomit ke Git karena berukuran besar. Pilihan yang tersedia:\n"
        "  1. unduh dari GitHub Release dan letakkan pada path di atas "
        "(lihat weights/README.md untuk tautan dan checksum), atau\n"
        "  2. setel SKIP_TRAINING=False untuk melatih ulang dari konfigurasi beku, atau\n"
        "  3. jalankan: python scripts/run_final_training.py --config configs/final_model_config.yaml"
    )
    print(f"Mode evaluasi: memuat weights final dari {FINAL_WEIGHTS_PATH}")
else:
    print("Mode reproduksi penuh: menjalankan pelatihan dari konfigurasi beku.")
    extra_kwargs = {
        "optimizer": FINAL_CONFIG["optimizer"], "lr0": FINAL_CONFIG["learning_rate"],
        "momentum": FINAL_CONFIG["momentum"], "weight_decay": FINAL_CONFIG["weight_decay"],
        "patience": FINAL_CONFIG["patience"], "flipud": FINAL_CONFIG.get("flipud", 0.0),
    }
    result = run_training(
        model_arch=FINAL_CONFIG["model_arch"],
        data_yaml=PREPARED_DIR / "data.yaml",
        output_project=PROJECT_ROOT / "runs" / "detect" / "final",
        run_name="final_model_notebook_rerun",
        image_size=FINAL_CONFIG["image_size"],
        batch_size=FINAL_CONFIG["batch_size"],
        epochs=FINAL_CONFIG["epochs"],
        device=detect_device(),
        seed=SEED,
        workers=FINAL_CONFIG["workers"],
        fraction=FINAL_CONFIG["fraction"],
        plots=True,
        validate=True,
        extra_train_kwargs=extra_kwargs,
    )
    FINAL_WEIGHTS_PATH = Path(result["best_weights"])
    print(f"Pelatihan selesai. Weights terbaik: {FINAL_WEIGHTS_PATH}")


### Catatan pelatihan final

Rekaman resmi proses pelatihan disimpan pada
`artifacts/reports/block15_final_training_summary.json`.

In [ ]:
with open(REPORTS_DIR / "block15_final_training_summary.json") as f:
    training_summary = json.load(f)

print(f"Git commit saat pelatihan : {training_summary['git_commit']}")
print(f"Hash manifest dataset     : {training_summary['dataset_manifest_hash']}")
print(f"Perangkat                 : {training_summary['device']}")
print(f"Jumlah epoch              : {training_summary['config_used']['epochs']}")
print(f"Durasi                    : {training_summary['duration_seconds'] / 3600:.2f} jam")
print(f"Validasi muat proses bersih: {'LULUS' if training_summary['clean_process_load_validation']['success'] else 'GAGAL'}")

### Kurva pelatihan

Kurva berikut dibaca langsung dari `results.csv` yang dihasilkan proses
pelatihan, bukan dari angka yang diketik ulang.

In [ ]:
results_csv = PROJECT_ROOT / "runs" / "detect" / "final" / "model_100epoch" / "results.csv"
if not results_csv.exists():
    results_csv = REPORTS_DIR / "final_training_history.csv"
assert results_csv.exists(), f"Riwayat pelatihan tidak ditemukan pada {results_csv}"
print(f"Sumber riwayat pelatihan: {results_csv.relative_to(PROJECT_ROOT)}")

history = {}
with open(results_csv) as f:
    for row in csv.DictReader(f):
        for k, v in row.items():
            history.setdefault(k.strip(), []).append(float(v))

fig_path = plot_training_curves(history, FIGURES_DIR / "training_curves.png")
plt.figure(figsize=(14, 5))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

best_epoch = max(range(len(history["metrics/mAP50(B)"])), key=lambda i: history["metrics/mAP50(B)"][i])
print(f"mAP@0.5 tertinggi  : {history['metrics/mAP50(B)'][best_epoch]:.4f} pada epoch {int(history['epoch'][best_epoch])}")
print(f"mAP@0.5 epoch akhir: {history['metrics/mAP50(B)'][-1]:.4f}")


Kurva di atas kami baca sebagai berikut. Komponen *loss* pada data latih menurun
konsisten sepanjang 50 *epoch* tanpa lonjakan yang menandakan ketidakstabilan.
Pada sisi validasi, kurva mAP@0.5 meningkat tajam di fase awal lalu melandai
pada sepertiga terakhir pelatihan.

Pelandaian itu menunjukkan bahwa perolehan tambahan dari *epoch* berikutnya
semakin kecil pada konfigurasi ini. Yang tidak kami temukan adalah pola
penurunan metrik validasi yang disertai penurunan *loss* latih secara
bersamaan, sehingga kami tidak mengklaim adanya *overfitting* berdasarkan data
yang kami punya.


## 14. Evaluasi

Model final sudah terlatih, dan sekarang kami mengukurnya. Evaluasi kami
jalankan melalui `scripts/evaluate.py` sebagai *subprocess* tersendiri.

Pemisahan proses ini bukan preferensi gaya. Uji reproduksi pada lingkungan
bersih menemukan bahwa menjalankan `val()` bawaan Ultralytics bersama
pengumpulan prediksi untuk *F1* lokal di dalam satu proses yang sama merusak
kondisi internal *backend* MPS pada perangkat ini. Karena itu kami memisahkan
setiap tahap ke proses masing-masing.

Perlu kami tegaskan sekali lagi bahwa data uji tidak pernah kami pakai untuk
penyetelan apa pun. Seluruh evaluasi pada notebook ini berjalan di atas *split*
validasi.


In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts" / "evaluate.py"),
     "--weights", str(FINAL_WEIGHTS_PATH),
     "--split", "valid",
     "--prepared-dir", str(PREPARED_DIR),
     "--conf-threshold", "0.25"],
    check=True, cwd=PROJECT_ROOT,
)

with open(REPORTS_DIR / "evaluation_valid.json") as f:
    eval_report = json.load(f)

### Hasil akhir

Sebelum menyebut angkanya, kami harus menyelesaikan satu persoalan. Regulasi
menyebut dua metrik penilaian, yaitu mAP@50 dan *F1-Score*, tanpa merinci cara
menghitungnya. Karena itu kami perlu menyatakan konvensi yang kami pakai secara
eksplisit, supaya juri dapat mereproduksi angka kami persis.

Untuk mAP@0.5 kami mengambil nilainya langsung dari `model.val()` bawaan
Ultralytics. Untuk *F1-Score* kami menghitung rata-rata antar kelas (*macro*)
dari kurva *F1* per kelas, diambil pada satu *confidence threshold* yang
memaksimalkan rata-rata tersebut. Dan kami memakai ambang NMS 0,5, bukan nilai
bawaan 0,7, berdasarkan pencarian pada *split* validasi yang kami dokumentasikan
di `artifacts/reports/inference_tuning_nms.json`.

Seluruh angka yang kami laporkan memakai konfigurasi di atas. Skrip
`scripts/compute_official_metrics.py` menghasilkan angka ini dan dapat
dijalankan ulang untuk verifikasi.


In [ ]:
official_path = REPORTS_DIR / "official_metrics_valid.json"
if not official_path.exists():
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "compute_official_metrics.py"),
         "--split", "valid"],
        check=True, cwd=PROJECT_ROOT,
    )

with open(official_path) as f:
    official = json.load(f)

lb = official["metrik_dilaporkan"]
det = official["rincian"]

print("=== METRIK YANG DILAPORKAN (split validasi) ===")
print(f"  mAP@50   : {lb['mAP50_persen']:.2f}%")
print(f"  F1-Score : {lb['f1_persen']:.2f}%")
print()
print("=== Rincian ===")
print(f"  ambang NMS IoU              : {official['konfigurasi_inferensi']['nms_iou']}")
print(f"  titik operasi confidence    : {det['confidence_titik_operasi']:.4f}")
print(f"  precision di titik operasi  : {det['precision_titik_operasi']:.4f}")
print(f"  recall di titik operasi     : {det['recall_titik_operasi']:.4f}")
print(f"  mAP@0.5:0.95                : {det['mAP50_95']:.4f}")

model_size_mb = FINAL_WEIGHTS_PATH.stat().st_size / (1024 * 1024)
print()
print(f"  ukuran model                : {model_size_mb:.2f} MB")
print(f"  ukuran citra masukan        : {FINAL_CONFIG['image_size']}")
print(f"  jumlah epoch                : {FINAL_CONFIG['epochs']}")

# Variabel di bawah dipakai bagian-bagian berikutnya. Metrik native berasal
# dari evaluate.py, sedangkan `local` adalah metrik diagnostik sekunder.
native = eval_report["native_metrics"]
local = eval_report["local_f1_metrics"]
per_class_ap = native["per_class_AP50"]

### Koreksi metodologi yang perlu kami nyatakan terbuka

Ada satu hal yang perlu kami sampaikan sebelum pembaca membandingkan angka di
laporan ini dengan versi sebelumnya. Kami semula melaporkan *F1* memakai
implementasi lokal dengan rata-rata *micro* pada *confidence threshold* tetap
0,25, yang menghasilkan angka sekitar 0,33. Belakangan kami sadari itu bukan
konvensi pelaporan yang lazim untuk model deteksi.

Kami menemukannya bukan dari perbandingan dengan pihak lain, melainkan dari
kejanggalan internal laporan kami sendiri. Ultralytics melaporkan *precision*
dan *recall* masing-masing sekitar 0,62, padahal *F1* secara matematis selalu
berada di antara kedua nilai itu. *F1* sebesar 0,33 karenanya mustahil untuk
pasangan angka tersebut, dan kejanggalan itulah yang memicu penelusuran
kami.

Penyebabnya adalah pilihan rata-rata *micro*. Dataset ini sangat timpang, dan
yang penting: kelas langka justru berperforma paling baik, sedangkan kelas
paling banyak justru paling buruk. Narrow brown dengan 222 *instance*
memperoleh AP 0,9631, sedangkan Brown spot dengan 5.010 *instance* hanya
0,2909. Pada rata-rata *micro*, Brown spot mendominasi dan menarik nilai
gabungan turun. Pada rata-rata *macro*, setiap kelas berbobot sama.

Rincian lengkap koreksi ini, termasuk mengapa nilai 0,6181 yang dipakai dan
bukan 0,6321 yang sedikit lebih tinggi, tersedia pada
[`artifacts/audit/metrics_methodology.md`](../artifacts/audit/metrics_methodology.md).

Kami tidak menghapus metrik lokal tersebut. Kami menurunkan statusnya menjadi
metrik diagnostik sekunder dan tetap dipakai pada analisis sensitivitas *threshold*
di Bagian 14.1, karena sifatnya deterministik dan berguna untuk melihat
perilaku model dari sisi pesimistis.

### 14.1 Sensitivitas terhadap *Confidence Threshold*

Satu angka tunggal tidak memberi tahu kami bagaimana model berperilaku ketika
ambang keyakinannya digeser, padahal itulah yang menentukan apakah model
berguna di lapangan. Karena itu kami memetakan perilakunya pada rentang
*threshold* yang luas.

Kami perlu memagari bagian ini dengan dua catatan agar tidak disalahbaca.
Pertama, metrik yang kami pakai di sini berstatus diagnostik sekunder, bukan
*F1* yang kami laporkan. Nilainya memakai rata-rata *micro*, sehingga sistematis
lebih rendah daripada *F1* macro pada bagian sebelumnya; keduanya mengukur hal
yang berbeda dan tidak boleh dibandingkan langsung.

Kedua, analisis ini kami hitung ulang dari prediksi yang sudah tersimpan, tanpa
menjalankan inferensi ulang. Karena tahap yang tidak deterministik adalah
inferensinya, bukan pencocokannya, hasil pada bagian ini bersifat deterministik
dan dapat direproduksi persis.

Kami juga tidak pernah memilih *threshold* berdasarkan data uji. Seluruh
analisis di sini berjalan pada *split* validasi.


In [ ]:
threshold_report_path = REPORTS_DIR / "threshold_sensitivity_valid.json"
if not threshold_report_path.exists():
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "threshold_sensitivity.py"),
         "--split", "valid", "--prepared-dir", str(PREPARED_DIR)],
        check=True, cwd=PROJECT_ROOT,
    )

with open(threshold_report_path) as f:
    threshold_report = json.load(f)

rows = threshold_report["rows"]
print(f"{'Threshold':>10s} {'Precision':>10s} {'Recall':>9s} {'F1 lokal':>9s} {'TP':>7s} {'FP':>7s} {'FN':>7s}")
for r in rows:
    print(
        f"{r['threshold']:10.2f} {r['precision']:10.4f} {r['recall']:9.4f} {r['f1']:9.4f} "
        f"{r['true_positives']:7d} {r['false_positives']:7d} {r['false_negatives']:7d}"
    )

best = threshold_report["best_f1_row"]
print(f"\nF1 lokal tertinggi: {best['f1']:.4f} pada threshold {best['threshold']:.2f}")

In [ ]:
fig_path = plot_threshold_sensitivity(rows, FIGURES_DIR / "threshold_sensitivity_valid.png")
plt.figure(figsize=(10, 6))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

Kami membaca tiga pola dari tabel dan grafik di atas.

Yang pertama, *precision* dan *recall* bergerak berlawanan secara tajam. Pada
*threshold* 0,05 *precision* hanya 0,2339 sementara *recall* 0,3331, sedangkan
pada *threshold* 0,90 *precision* mencapai 0,9744 tetapi *recall* jatuh ke
0,0078. Artinya ketika model cukup yakin, prediksinya hampir selalu benar, namun
jumlah objek yang berani ia tandai menjadi sangat sedikit.

Yang kedua, *F1* lokal tertinggi berada pada *threshold* 0,15 dengan nilai
0,3479, sedikit di atas nilai pada *threshold* 0,25 yang kami pakai sebagai
acuan pelaporan, yaitu 0,3326. Selisihnya kecil, sehingga pemilihan 0,25 sebagai
acuan tidak mengubah kesimpulan kami secara berarti.

Yang ketiga, dan ini yang paling menentukan arah pekerjaan kami selanjutnya,
faktor pembatas utamanya adalah *recall*, bukan *precision*. Nilai *recall*
tertinggi pada seluruh rentang hanya 0,3331, yang berarti sekitar dua pertiga
objek *ground truth* tidak pernah terpasangkan dengan prediksi pada kriteria IoU
minimal 0,5, bahkan ketika kami menurunkan *threshold* sampai 0,05. Temuan ini
yang mengarahkan analisis kesalahan kami pada Bagian 16 untuk berfokus pada
*false negative*, dan sejalan dengan dominasi objek kecil yang kami temukan pada
Bagian 6.5.

Sekali lagi kami tegaskan bahwa nilai *F1* pada tabel ini berasal dari
implementasi lokal kami, sehingga tidak dapat disebut sebagai skor resmi lomba.


### 14.2 Hasil pada *Split* Test

Seluruh penyetelan pada project ini kami lakukan hanya menggunakan *split*
validasi. Kami tidak pernah memakai *split* test untuk memilih *hyperparameter*,
menentukan kapan pelatihan berhenti, maupun memilih *threshold*.

Evaluasi berikut kami jalankan satu kali saja, setelah model kami bekukan dan
seluruh keputusan selesai kami ambil. Tujuan kami adalah memperoleh estimasi
generalisasi yang jujur pada data yang belum pernah memengaruhi keputusan apa
pun, dan kami tidak memakai angka ini untuk mengubah model.


In [ ]:
test_report_path = REPORTS_DIR / "evaluation_test.json"
if not test_report_path.exists():
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "evaluate.py"),
         "--weights", str(FINAL_WEIGHTS_PATH), "--split", "test",
         "--prepared-dir", str(PREPARED_DIR), "--conf-threshold", "0.25"],
        check=True, cwd=PROJECT_ROOT,
    )

official_test_path = REPORTS_DIR / "official_metrics_test.json"
if not official_test_path.exists():
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "compute_official_metrics.py"),
         "--split", "test"],
        check=True, cwd=PROJECT_ROOT,
    )

with open(test_report_path) as f:
    test_report = json.load(f)
with open(official_test_path) as f:
    official_test = json.load(f)

test_native = test_report["native_metrics"]
lb_test = official_test["metrik_dilaporkan"]

print("=== METRIK YANG DILAPORKAN, perbandingan dua split ===")
print(f"{'Metrik':14s} {'Valid':>10s} {'Test':>10s} {'Selisih':>10s}")
print(f"{'mAP@50 (%)':14s} {lb['mAP50_persen']:10.2f} {lb_test['mAP50_persen']:10.2f} {lb_test['mAP50_persen']-lb['mAP50_persen']:+10.2f}")
print(f"{'F1-Score (%)':14s} {lb['f1_persen']:10.2f} {lb_test['f1_persen']:10.2f} {lb_test['f1_persen']-lb['f1_persen']:+10.2f}")
print()
print("=== Rincian tambahan ===")
print(f"{'mAP@0.5:0.95':14s} {det['mAP50_95']:10.4f} {official_test['rincian']['mAP50_95']:10.4f}")
print(f"{'conf operasi':14s} {det['confidence_titik_operasi']:10.4f} {official_test['rincian']['confidence_titik_operasi']:10.4f}")

In [ ]:
print(f"{'Kelas':28s} {'AP valid':>9s} {'AP test':>9s} {'Selisih':>9s}")
for cls, ap_v in sorted(native["per_class_AP50"].items(), key=lambda kv: kv[1], reverse=True):
    ap_t = test_native["per_class_AP50"][cls]
    print(f"{cls:28s} {ap_v:9.4f} {ap_t:9.4f} {ap_t - ap_v:+9.4f}")

Hasilnya sangat dekat dengan validasi. Nilai mAP@0.5 turun tipis sebesar
0,0132, sedangkan mAP@0.5:0.95 justru naik 0,0093.

Kedekatan inilah yang menurut kami paling penting dari seluruh bagian evaluasi.
Karena *split* test tidak pernah menyentuh proses pengambilan keputusan kami,
selisih sekecil itu menunjukkan bahwa performa model tidak bergantung pada
*split* validasi tertentu. Dengan kata lain, kami tidak menemukan indikasi bahwa
konfigurasi final ter-*overfit* terhadap data validasi meskipun kami menjalankan
21 percobaan di atasnya.

Urutan kelasnya pun konsisten pada kedua *split*. Narrow brown tetap tertinggi
(0,9809 pada test) dan Brown spot tetap terendah (0,2582). Konsistensi peringkat
ini memperkuat analisis kami pada Bagian 15 dan 17, karena menunjukkan bahwa
perbedaan performa antar kelas merupakan sifat yang stabil, bukan kebetulan pada
satu *split*.

Satu catatan terakhir, *split* test lebih timpang distribusinya dengan rasio
31,1 kali dan memiliki proporsi objek kecil tertinggi sebesar 46,8 persen,
sehingga penurunan tipis mAP@0.5 di sana justru sejalan dengan karakteristik
datanya sendiri.


## 15. Hasil Per Kelas

Angka agregat pada bagian sebelumnya memberi kami satu bilangan untuk seluruh
model, dan justru di situ bahayanya: rata-rata dapat menyembunyikan perbedaan
besar antar kelas. Sebuah model bisa terlihat memadai secara keseluruhan
padahal praktis buta terhadap satu atau dua penyakit.

Karena itu kami membongkar hasil tersebut per kelas. Pertanyaan yang kami bawa
ke bagian ini adalah kelas mana yang sudah dikenali model dengan baik, kelas
mana yang belum, dan seberapa lebar jaraknya.


In [ ]:
per_class_ap = native["per_class_AP50"]
fig_path = plot_per_class_ap(per_class_ap, FIGURES_DIR / "per_class_ap.png")
plt.figure(figsize=(10, 6))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

print(f"{'Kelas':28s} {'AP@0.5':>8s} {'Instance train':>15s}")
for cls, ap in sorted(per_class_ap.items(), key=lambda kv: kv[1], reverse=True):
    print(f"{cls:28s} {ap:8.4f} {train_imb.per_class_instances[cls]:15d}")

Sebaran hasilnya jauh lebih lebar daripada yang tersirat dari angka agregat.
Kelas dengan performa tertinggi adalah Narrow brown, False smut, Leaf roller,
dan Healthy, seluruhnya di atas AP@0.5 sebesar 0,87. Sementara kelas dengan
performa terendah adalah Brown spot, Leaf scald, dan Bacterial leaf blight,
seluruhnya di bawah 0,39.

Yang membuat kami berhenti sejenak adalah arah hubungannya. Narrow brown justru
merupakan kelas dengan jumlah *instance* paling sedikit pada *split* latih, yaitu
222, tetapi memperoleh AP@0.5 tertinggi. Sebaliknya Brown spot memiliki jumlah
*instance* terbanyak, yaitu 5.010, namun memperoleh AP@0.5 terendah.

Pola itu berlawanan dengan dugaan umum bahwa lebih banyak data berarti performa
lebih baik, dan cukup penting untuk tidak kami lewatkan begitu saja. Kami
memeriksanya secara khusus pada Bagian 17.


### 15.1 Perbandingan dengan Metrik Lokal per Kelas

Sebelum menarik kesimpulan dari AP@0.5 saja, kami ingin memeriksanya dengan cara
ukur yang berbeda. AP@0.5 mengintegrasikan seluruh kurva *precision-recall*,
sedangkan metrik lokal kami hitung pada satu *confidence threshold* tetap.
Keduanya menjawab pertanyaan yang berbeda, dan justru karena itu kami membacanya
berdampingan: bila keduanya sepakat, keyakinan kami bertambah; bila berbeda jauh,
ada sesuatu yang perlu dijelaskan.


In [ ]:
with open(REPORTS_DIR / "block13_error_analysis.json") as f:
    ea = json.load(f)

pcr = ea["per_class_precision_recall"]
print(f"Metrik lokal dihitung pada confidence threshold {ea['confidence_threshold']}\n")
print(f"{'Kelas':28s} {'AP@0.5':>8s} {'P lokal':>9s} {'R lokal':>9s} {'TP':>6s} {'FN':>6s} {'Instance train':>15s}")
for cls, ap in sorted(per_class_ap.items(), key=lambda kv: kv[1], reverse=True):
    m = pcr.get(cls, {})
    print(
        f"{cls:28s} {ap:8.4f} {m.get('precision', 0):9.3f} {m.get('recall', 0):9.3f} "
        f"{m.get('tp', 0):6d} {m.get('fn', 0):6d} {train_imb.per_class_instances[cls]:15d}"
    )

Perbandingan ini memunculkan satu ketidaksesuaian yang perlu kami nyatakan
terbuka. Narrow brown memperoleh AP@0.5 tertinggi sebesar 0,9631, tetapi
*recall* lokalnya hanya 0,213.

Kedua angka itu sebenarnya tidak bertentangan, karena dihitung dengan prosedur
berbeda: yang satu mengintegrasikan kurva penuh, yang lain mencocokkan pada satu
*threshold* tetap. Namun perbedaan sebesar ini menjadi peringatan bagi kami
sendiri. AP@0.5 untuk kelas dengan jumlah *instance* sangat sedikit, yaitu hanya
75 kotak *ground truth* pada *split* validasi, perlu dibaca hati-hati, sebab
sedikit prediksi yang kebetulan terurut dengan baik dapat menghasilkan AP tinggi
tanpa berarti model menemukan sebagian besar objeknya.

Pola sebaliknya kami lihat pada False smut dan Leaf roller, yang memiliki AP
tinggi sekaligus *recall* lokal tinggi, yaitu 0,942 dan 0,885. Pada kedua kelas
ini, performa tinggi konsisten pada kedua cara pengukuran, sehingga kami lebih
yakin menyebutnya sebagai keberhasilan yang sesungguhnya.


## 16. Analisis Kesalahan

Mengetahui kelas mana yang lemah belum memberi tahu kami mengapa kelas itu
lemah. Model bisa saja gagal karena tidak melihat objeknya sama sekali, atau
justru melihatnya tetapi salah menamainya, dan kedua kemungkinan itu menuntut
perbaikan yang berbeda.

Untuk memisahkannya, kami memakai hasil `scripts/run_error_analysis.py`, yang
mengelompokkan setiap kesalahan menjadi empat kategori: *true positive*, salah
kelas, *false positive* terhadap latar belakang, dan *false negative*. Komposisi
keempatnya yang akan kami baca sebagai petunjuk arah perbaikan.


In [ ]:
error_analysis = ea
counts = error_analysis["counts"]
total_errors = counts["class_confusions"] + counts["background_false_positives"] + counts["false_negatives"]

print("Rincian hasil pencocokan (class-agnostic terlebih dahulu, lalu diklasifikasi):")
for k, v in counts.items():
    print(f"  {k:34s}: {v:6d}")

print(f"\nKomposisi kesalahan (total {total_errors}):")
for k in ("false_negatives", "background_false_positives", "class_confusions"):
    print(f"  {k:34s}: {counts[k] / total_errors:6.1%}")

small = error_analysis["small_object_miss_analysis"]
print(f"\nMedian luas bbox keseluruhan     : {small['overall_median_gt_area_px2']} px2")
print(f"Median luas bbox yang terlewat   : {small['false_negative_median_area_px2']} px2")
print(f"Rasio                            : {small['false_negative_median_area_px2'] / small['overall_median_gt_area_px2']:.2f}")

crowd = error_analysis["crowded_vs_sparse_scene_fn_rate"]
print(f"\nRasio false negative scene padat : {crowd['crowded_scene_fn_rate']}")
print(f"Rasio false negative scene jarang: {crowd['sparse_scene_fn_rate']}")

Komposisi kesalahannya memberi arah yang cukup jelas. Kesalahan kami didominasi
oleh *false negative*, yaitu objek yang ada pada *ground truth* tetapi tidak
ditemukan model, bukan oleh salah kelas. Salah kelas justru merupakan porsi
terkecil.

Bagi kami ini kabar yang cukup melegakan sekaligus menantang. Masalah utama
model ini bukan membedakan penyakit satu dengan lainnya, melainkan menemukan
objeknya terlebih dahulu. Temuan ini konsisten dengan analisis *threshold* pada
Bagian 14.1, yang juga menempatkan *recall* sebagai faktor pembatas.


### 16.1 Pasangan Kelas yang Tertukar

Meskipun salah kelas merupakan porsi terkecil, polanya tetap informatif.

In [ ]:
print(f"{'Kelas sebenarnya':28s} {'Diprediksi sebagai':28s} {'Jumlah':>7s}")
for pair in error_analysis["top_confused_pairs"][:8]:
    print(f"{pair['true_class']:28s} {pair['predicted_class']:28s} {pair['count']:7d}")

Kebingungan terbesar kami temukan pada pasangan Blast dan Bacterial panicle
blight, yang tertukar pada kedua arah. Pasangan berikutnya adalah Brown spot
yang diprediksi sebagai Blast. Ketiganya merupakan penyakit yang menimbulkan
lesi kecokelatan pada jaringan tanaman, sehingga kemiripan visualnya masuk akal
secara domain dan tidak mengejutkan kami.

Meski begitu kami perlu mencatat bahwa jumlah absolutnya kecil, yaitu 17 dan 12
kejadian untuk kedua pasangan teratas. Kami menyebutkannya sebagai pola yang
terbaca, bukan sebagai masalah utama model.


In [ ]:
fp_examples = sorted((PROJECT_ROOT / "artifacts" / "figures" / "error_analysis").glob("bg_fp_*.jpg"))[:2]
fn_examples = sorted((PROJECT_ROOT / "artifacts" / "figures" / "error_analysis").glob("fn_*.jpg"))[:2]
examples = [(p, "False positive latar belakang") for p in fp_examples]
examples += [(p, "False negative") for p in fn_examples]

if examples:
    fig, axes = plt.subplots(1, len(examples), figsize=(5 * len(examples), 5))
    if len(examples) == 1:
        axes = [axes]
    for ax, (path, label) in zip(axes, examples):
        ax.imshow(mpimg.imread(path))
        ax.set_title(label, fontsize=10)
        ax.axis("off")
    plt.suptitle("Contoh kesalahan model pada split validasi", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Belum ada figur analisis kesalahan. Jalankan scripts/run_error_analysis.py.")

Dua pola kesalahan menonjol dari analisis ini, dan keduanya kami dukung dengan
angka.

Yang pertama, objek yang terlewat cenderung lebih kecil daripada rata-rata.
Median luas *bounding box* yang gagal terdeteksi jauh di bawah median luas
seluruh *bounding box*. Yang membuat temuan ini kuat menurut kami adalah bahwa
ia berasal dari kesalahan aktual, bukan sekadar dugaan yang kami tarik dari
angka agregat, dan ia konsisten dengan dominasi objek kecil yang kami temukan
jauh sebelumnya pada Bagian 6.5.

Yang kedua, adegan padat memiliki rasio *false negative* lebih tinggi daripada
adegan jarang. Ini konsisten dengan pola pertama, karena objek kecil yang saling
berdekatan memang merupakan kasus tersulit, persis seperti citra Brown spot yang
kami tampilkan pada Bagian 6.11.


### 16.2 Perbandingan *Ground Truth* dengan Prediksi

Angka sudah cukup banyak, dan kami ingin memperlihatkan langsung seperti apa
wujud kesalahan itu. Bagian berikut menampilkan citra utuh dengan anotasi
sebenarnya di sisi kiri dan prediksi model di sisi kanan.

Karena menampilkan contoh visual selalu membuka peluang memilih yang
menguntungkan diri sendiri, kami menyatakan aturan pemilihannya lebih dulu.
Untuk kelompok keberhasilan kami mengambil citra dengan minimal satu *true
positive*, tanpa *false negative*, dan tanpa *false positive*, lalu mengurutkannya
menurun menurut jumlah *true positive*. Untuk kelompok kegagalan kami mengambil
citra dengan jumlah kesalahan terbanyak.

Kami menegaskan bahwa kelompok keberhasilan merupakan ilustrasi kemampuan model
pada kondisi yang menguntungkan, bukan representasi statistik dari keseluruhan
performa. Proporsi sebenarnya kami laporkan tepat di bawah gambar.


In [ ]:
examples_report_path = REPORTS_DIR / "prediction_examples_valid.json"
if not examples_report_path.exists():
    subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "scripts" / "visualize_predictions_vs_truth.py"),
         "--split", "valid", "--prepared-dir", str(PREPARED_DIR)],
        check=True, cwd=PROJECT_ROOT,
    )

with open(examples_report_path) as f:
    examples_report = json.load(f)

n_perfect = examples_report["images_with_perfect_detection"]
n_total = examples_report["total_images_with_ground_truth"]
n_error = examples_report["images_with_any_error"]

print(f"Citra dengan deteksi sempurna     : {n_perfect} dari {n_total} ({n_perfect / n_total:.1%})")
print(f"Citra dengan minimal satu kesalahan: {n_error} dari {n_total} ({n_error / n_total:.1%})")
print(f"Confidence threshold              : {examples_report['confidence_threshold']}")

In [ ]:
for label, judul in (("success", "Contoh prediksi benar"), ("failure", "Contoh kegagalan")):
    path = FIGURES_DIR / f"{label}_cases_valid.png"
    if path.exists():
        plt.figure(figsize=(12, 16))
        plt.imshow(mpimg.imread(path))
        plt.axis("off")
        plt.title(judul)
        plt.show()

Pada kelompok keberhasilan, model menemukan seluruh lesi Blast dan Tungro pada
citra yang bersangkutan, dengan kotak yang berimpit rapat terhadap anotasi
sebenarnya dan *confidence* pada kisaran 0,31 sampai 0,84. Kami membacanya
sebagai bukti bahwa ketika gejala tampak cukup jelas dan kontrasnya memadai,
model mampu melokalisasi beberapa objek sekaligus dalam satu citra.

Namun kami tidak ingin pembaca berhenti pada gambar itu saja. Hanya sekitar 22
persen citra validasi yang seluruh objeknya terdeteksi tanpa kesalahan,
sedangkan sisanya memiliki minimal satu *false negative* atau *false positive*.
Angka inilah gambaran realistisnya, bukan contoh visual di atas.


## 17. Hubungan Karakteristik Data dengan Performa

Setelah melihat kelas mana yang lemah dan bentuk kesalahannya, muncul dugaan
yang wajar: barangkali kelas dengan data lebih banyak otomatis memperoleh
performa lebih baik. Dugaan ini sering diterima begitu saja, dan bila benar,
arah perbaikannya jelas, yaitu menambah data pada kelas minoritas.

Kami tidak ingin menerimanya tanpa uji. Bagian ini menghadapkan dugaan tersebut
pada data kami sendiri, dengan memeriksa korelasi antara jumlah anotasi per
kelas dan performa yang dicapai kelas itu.


In [ ]:
fig_path = plot_instances_vs_performance(
    train_imb.per_class_instances, per_class_ap, FIGURES_DIR / "instances_vs_ap.png"
)
plt.figure(figsize=(10, 7))
plt.imshow(mpimg.imread(fig_path))
plt.axis("off")
plt.show()

import statistics as st

classes = [c for c in CANONICAL_CLASSES if c in per_class_ap]
xs = [train_imb.per_class_instances[c] for c in classes]
ys = [per_class_ap[c] for c in classes]
try:
    corr = st.correlation(xs, ys)
    print(f"Korelasi Pearson antara jumlah instance dan AP@0.5: {corr:.4f}")
except Exception as exc:
    print(f"Korelasi tidak dapat dihitung: {exc}")

Korelasi Pearson antara jumlah *instance* pada *split* latih dan AP@0.5 per
kelas bernilai sekitar -0,51, yaitu korelasi negatif dengan kekuatan sedang.
Arah hubungannya berlawanan dengan dugaan umum: pada dataset ini, kelas
dengan jumlah data lebih banyak justru cenderung memperoleh AP@0.5 lebih
rendah.

Kami perlu memasang tiga peringatan sebelum pembaca menafsirkan angka itu.
Pertama, hubungan ini bersifat observasional, bukan kausal, dan kami tidak
menjalankan eksperimen terkontrol untuk mengujinya. Kedua, korelasi kami
hitung hanya dari 11 titik data, sehingga sangat sensitif terhadap beberapa
kelas ekstrem dan tidak bisa kami anggap sebagai bukti kuat. Ketiga,
arah negatif ini kemungkinan besar merupakan gejala dari variabel lain yang
kebetulan berkorelasi dengan jumlah data, bukan bukti bahwa menambah data
merugikan.

Yang dapat disimpulkan secara aman hanyalah bahwa jumlah data per kelas
tidak cukup untuk menjelaskan perbedaan performa antar kelas pada kasus
ini, sehingga faktor lain perlu dipertimbangkan.

Faktor lain yang secara masuk akal dapat berkontribusi, dan sebagiannya
didukung temuan pada bagian sebelumnya:

- **Ukuran objek.** Brown spot berupa bercak kecil yang tersebar, dan kelas
  ini memiliki rasio *instance* per citra tertinggi, yaitu sekitar 4,5
  *instance* per citra. Kombinasi objek kecil dan adegan padat merupakan
  kondisi yang terbukti paling sulit pada Bagian 16.
- **Kekhasan visual.** Narrow brown dan False smut memiliki penampakan yang
  relatif khas, sedangkan beberapa penyakit bercak daun memiliki kemiripan
  visual satu sama lain.
- **Konsistensi anotasi.** Objek kecil yang banyak pada satu citra lebih
  rentan terhadap variasi cara anotasi dilakukan.

## 18. Kelebihan dan Keterbatasan

Setelah seluruh angka dipaparkan, kami merasa perlu berhenti sejenak untuk
menilai pekerjaan ini secara utuh, termasuk bagian yang tidak menguntungkan
bagi kami.

### Kelebihan pendekatan

Yang paling kami andalkan adalah kepatuhan yang dapat diverifikasi pada tingkat
kode. Larangan *external pretrained weights* tidak hanya kami nyatakan di
dokumen, tetapi kami tegakkan lewat fungsi yang menolak berjalan bila
dilanggar, ditambah `YOLO_OFFLINE` yang membuat setiap upaya pengunduhan gagal
secara keras.

Pola yang sama kami terapkan pada pemetaan canonical: kategori mentah yang
tidak dikenali akan menghentikan proses, sehingga perubahan pada dataset tidak
bisa lolos diam-diam. Penyiapan datanya sendiri deterministik dan tidak
merusak, karena dataset mentah tidak pernah kami ubah dan *manifest* dapat kami
hasilkan ulang secara identik.

Di sisi pemodelan, seluruh keputusan konfigurasi kami sandarkan pada 21
percobaan yang terdokumentasi lengkap dengan *seed*, *hyperparameter*, dan
*commit*, dan setiap tahap pekerjaan meninggalkan laporan terstruktur yang
dapat diperiksa ulang. Terakhir, kami melaporkan keterbatasan apa adanya,
termasuk ketidakstabilan *F1* lokal yang sebenarnya merugikan penyajian hasil
kami sendiri.

### Keterbatasan

Model ini kami latih dari nol, sehingga tidak mewarisi representasi visual umum
yang biasanya didapat dari *pretrained weights*. Ini konsekuensi aturan
kompetisi, bukan pilihan desain kami.

Pada sisi reproduksi, *backend* MPS bersifat nondeterministik. Operasi
`scatter_reduce_mps` dan `index_put_with_accumulate_mps` tidak memiliki
implementasi deterministik pada perangkat ini, sehingga kami tidak mengklaim
reproduksi bit per bit pada pelatihan. Terkait itu pula, *F1* lokal kami
bervariasi pada rentang 0,22 sampai 0,42 antar pengulangan pada *checkpoint*
yang sama, dan penyebab pastinya belum kami telusuri sampai tuntas.

Dari sisi hasil, performa antar kelas masih jauh dari merata: selisih AP@0.5
antara kelas terbaik dan terburuk melebihi 0,67. Pemilihan *hyperparameter*-nya
pun kami validasi pada skala penyaringan lalu kami terapkan pada skala penuh,
sehingga perilaku skala penuh hanya teramati langsung untuk konfigurasi final.

Tiga hal terakhir kami catat karena belum sempat kami kerjakan. Kandidat
duplikat berbasis *perceptual hash* belum kami periksa satu per satu secara
visual. Model belum kami uji pada data di luar dataset kompetisi, sehingga
kemampuan generalisasinya ke kondisi lapangan yang berbeda belum diketahui. Dan
seluruh pekerjaan ini berjalan pada satu laptop, yang membatasi ukuran model
sekaligus jumlah percobaan yang sanggup kami jalankan.


## 19. Reproducibility dan Audit

Karena juri akan menjalankan ulang pipeline ini, kami perlu menyatakan dengan
tepat sejauh mana hasil kami dapat direproduksi. Kami membedakannya menjadi
tiga tingkat agar klaim kami tidak melampaui bukti yang kami punya.

Tingkat pertama, prapemrosesan, sudah terverifikasi byte per byte. Menjalankan
ulang penyiapan dataset dengan *seed* yang sama menghasilkan *manifest* yang
identik.

Tingkat kedua, konfigurasi, juga terverifikasi. Setiap percobaan mencatat
*seed*, *hyperparameter*, arsitektur, *hash manifest*, dan *commit* Git yang
berlaku saat percobaan dijalankan.

Tingkat ketiga, reproduksi pelatihan bit per bit, tidak kami klaim. Penyebabnya
nondeterminisme *backend* MPS yang sudah kami jelaskan di bagian sebelumnya.
Dokumentasi resmi PyTorch sendiri menyatakan bahwa hasil yang sepenuhnya dapat
direproduksi tidak dijamin antar rilis, *commit*, maupun platform, dan sebagian
operasi memang tidak memiliki implementasi deterministik (PyTorch, 2026).

Risiko dari tingkat ketiga kami kurangi sebisanya melalui *seed* tetap,
konfigurasi yang dibekukan, *manifest* tetap, validasi pemuatan pada proses
bersih, *checksum* model, dan kode yang terversi.


In [ ]:
def sha256_of_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

with open(REPORTS_DIR / "final_model_metadata.json") as f:
    model_meta = json.load(f)

checksum_live = sha256_of_file(FINAL_WEIGHTS_PATH)

print("Tabel provenance")
print(f"{'Artefak':26s} {'Nilai'}")
print(f"{'Commit saat pelatihan':26s} {training_summary['git_commit']}")
print(f"{'Commit saat notebook ini':26s} {env['git_commit']}")
print(f"{'Hash manifest dataset':26s} {training_summary['dataset_manifest_hash']}")
print(f"{'Versi pemetaan kelas':26s} {model_meta['mapping_version']}")
print(f"{'SHA-256 weights (live)':26s} {checksum_live}")
print(f"{'SHA-256 weights (tercatat)':26s} {model_meta['weights_checksum_sha256']}")
print(f"{'Ukuran weights':26s} {FINAL_WEIGHTS_PATH.stat().st_size} byte")

assert checksum_live == model_meta["weights_checksum_sha256"], "Checksum weights tidak cocok dengan catatan."
print("\nChecksum cocok dengan metadata yang tercatat.")

Ada satu perbedaan pada tabel di atas yang perlu kami jelaskan, yaitu *commit*
saat pelatihan yang tidak sama dengan *commit* saat notebook ini dieksekusi. Hal
itu wajar, karena kami terus memperbarui dokumentasi setelah pelatihan selesai.

Yang justru harus kami pastikan adalah kode evaluasinya tidak berubah di antara
keduanya. Kami memverifikasinya melalui perbandingan `git diff` pada berkas
evaluasi, metrik, dan pelatihan, dan hasilnya kosong. *Checksum* modelnya pun
identik, sehingga bobot yang kami evaluasi memang bobot yang sama dengan yang
kami latih.


### Uji inferensi mandiri

Sebagai pemeriksaan terakhir, kami memuat *weights* dari awal, terlepas dari
kondisi proses pelatihan, lalu menjalankan inferensi pada lima citra validasi
yang kami pilih secara deterministik menggunakan *seed* global.

Kami sengaja tidak menyaring sampelnya berdasarkan keberhasilan deteksi, jadi
bila ada citra yang tidak menghasilkan satu deteksi pun, citra itu tetap kami
tampilkan apa adanya.


In [ ]:
from ultralytics import YOLO

inference_model = YOLO(str(FINAL_WEIGHTS_PATH))

valid_images = sorted((PREPARED_DIR / "valid" / "images").iterdir())
set_global_seed(SEED)
demo_paths = random.sample(valid_images, 5)

for path in demo_paths:
    results = inference_model.predict(str(path), verbose=False)
    boxes = results[0].boxes
    print(f"{path.name[:52]:54s} {len(boxes)} deteksi")
    for box in boxes:
        cls_name = CANONICAL_CLASSES[int(box.cls.item())]
        print(f"    {cls_name:28s} confidence {float(box.conf.item()):.3f}")

## 20. Kesimpulan

Kami menutup laporan ini dengan menjawab enam pertanyaan yang kami ajukan di
Bagian 2.

Yang berhasil kami bangun adalah pipeline deteksi penyakit tanaman padi untuk
11 kelas canonical, mulai dari audit dataset mentah sampai model terlatih
beserta seluruh jejak auditnya, memakai YOLOv8n yang dilatih sepenuhnya dari
nol tanpa *external pretrained weights*.

Pada *split* validasi, model mencapai mAP@0.5 sebesar 0,6401 dan *F1* macro
sebesar 0,6383. Ketika kami ujikan pada *split* test yang tidak pernah
menyentuh satu pun keputusan penyetelan, angkanya 0,6246 dan 0,6272, hanya
sekitar 1,5 poin di bawah hasil validasi. Kedekatan itu konsisten dengan
generalisasi yang cukup stabil pada kedua *split*, meskipun kami tidak dapat
menyimpulkan ketiadaan *overfitting* hanya dari satu evaluasi *held-out*.

Yang tidak bisa kami sembunyikan adalah ketimpangan antar kelas. Narrow brown
mencapai AP@0.5 sebesar 0,9717, sedangkan Brown spot hanya 0,3153. Selisih
lebih dari 0,65 poin itu terlalu besar untuk diabaikan, dan justru menjadi
bagian paling menarik dari analisis kami.

Sumber ketimpangannya ternyata bukan jumlah data, dan ini berlawanan dengan
dugaan awal kami sendiri. Korelasi antara jumlah *instance* dan AP@0.5 justru
negatif. Brown spot punya data terbanyak tetapi performa terburuk, sedangkan
Narrow brown sebaliknya. Ketika kami telusuri lebih jauh, penjelasan yang
lebih masuk akal datang dari geometri dan kepadatan objek: 63,5 persen
kesalahan model berupa *false negative*, objek yang terlewat median luasnya
sekitar setengah median keseluruhan, dan pada adegan padat rasio *false
negative* mencapai 0,7981. Brown spot kebetulan adalah kelas yang gejalanya
berupa puluhan bercak kecil berdesakan dalam satu citra.

Dengan kata lain, keterbatasan performa model kami tidak dapat dijelaskan
hanya oleh jumlah data. Geometri objek, kepadatan adegan, dan kapasitas model
yang dilatih dari nol perlu dibaca bersama-sama. Kami menegaskan bahwa
hubungan ini bersifat observasional, bukan kausal, karena kami tidak
menjalankan eksperimen terkontrol untuk mengujinya.

Kekuatan utama pekerjaan ini, menurut kami, bukan pada angkanya melainkan
pada keterlacakannya. Kepatuhan aturan kami tegakkan di tingkat kode sehingga
pelanggaran gagal secara terlihat, setiap keputusan konfigurasi dapat
ditelusuri ke percobaan yang tercatat, dan kami melaporkan keterbatasan apa
adanya termasuk yang merugikan penyajian hasil.

Kelemahan utamanya tetap jelas: performa rendah pada kelas dengan objek kecil
yang padat, metrik diagnostik yang tidak stabil pada perangkat yang kami
pakai, dan belum adanya validasi di luar dataset kompetisi.

Untuk penelitian berikutnya, bukti yang kami kumpulkan mengarah pada satu
prioritas yang cukup spesifik: menangani objek kecil secara khusus, misalnya
melalui resolusi masukan yang lebih tinggi sejak tahap pelatihan atau
strategi *tiling*, disertai verifikasi konsistensi anotasi pada kelas dengan
kepadatan objek tertinggi.

## 21. Daftar Pustaka

Kami hanya menambahkan sitasi untuk pernyataan yang benar-benar bersumber dari
literatur atau dokumentasi resmi. Pernyataan yang berasal dari aturan kompetisi
maupun dari hasil eksekusi kami sendiri tidak kami beri sitasi, karena sumbernya
adalah dokumen lomba dan artefak pada repository ini. Seluruh metadata di bawah diverifikasi terhadap halaman penerbit
atau dokumentasi resmi, bukan dikutip dari ingatan.

### Sumber ilmiah

Buda, M., Maki, A., & Mazurowski, M. A. (2018). A systematic study of the
class imbalance problem in convolutional neural networks. *Neural Networks,
106*, 249-259. https://doi.org/10.1016/j.neunet.2018.07.011

Feng, Q., Xu, X., & Wang, Z. (2023). Deep learning-based small object
detection: A survey. *Mathematical Biosciences and Engineering, 20*(4),
6551-6590. https://doi.org/10.3934/mbe.2023282

Lin, T.-Y., Maire, M., Belongie, S., Hays, J., Perona, P., Ramanan, D.,
Dollar, P., & Zitnick, C. L. (2014). Microsoft COCO: Common objects in
context. In D. Fleet, T. Pajdla, B. Schiele, & T. Tuytelaars (Eds.),
*Computer Vision - ECCV 2014* (Lecture Notes in Computer Science, Vol. 8693,
pp. 740-755). Springer. https://doi.org/10.1007/978-3-319-10602-1_48

Loshchilov, I., & Hutter, F. (2019). Decoupled weight decay regularization.
*7th International Conference on Learning Representations (ICLR 2019)*.
https://arxiv.org/abs/1711.05101

Mohanty, S. P., Hughes, D. P., & Salathe, M. (2016). Using deep learning for
image-based plant disease detection. *Frontiers in Plant Science, 7*, 1419.
https://doi.org/10.3389/fpls.2016.01419

Shorten, C., & Khoshgoftaar, T. M. (2019). A survey on image data
augmentation for deep learning. *Journal of Big Data, 6*, 60.
https://doi.org/10.1186/s40537-019-0197-0

### Sumber dokumentasi framework

PyTorch. (2026). *Reproducibility*. PyTorch documentation.
https://docs.pytorch.org/docs/stable/notes/randomness.html

Ultralytics. (2026). *Explore Ultralytics YOLOv8*. Ultralytics
documentation. https://docs.ultralytics.com/models/yolov8/

### Sumber kompetisi

Panitia TELEPATI 8.0, HIMATEL Politeknik Negeri Bandung. (2026). *Guidebook
dan regulasi AgriData Intelligence Race*. Dokumen kompetisi, tidak
dipublikasikan.

### Catatan mengenai penggunaan sitasi

Rincian pemeriksaan setiap sitasi, termasuk kalimat mana yang didukungnya,
tersedia pada [`references/README.md`](../references/README.md).

## 22. Lampiran Teknis

Lampiran ini kami sediakan bagi pembaca yang ingin menjalankan ulang atau
menelusuri sendiri isi repository kami.

### Struktur repository

```
agridata/
  configs/          konfigurasi eksperimen dan konfigurasi final beku
  src/agridata/     paket inti: dataset, training, metrics, analysis, visualization
  scripts/          titik masuk CLI untuk setiap tahap pipeline
  notebooks/        notebook submission ini
  artifacts/        audit, laporan, figur, log eksperimen
  tests/            uji unit
  weights/          informasi rilis model
```

### Perintah reproduksi

```bash
python3.11 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt

python scripts/prepare_dataset.py --dataset-root "Telepati 8.0 Datasets" --output-dir data/prepared --seed 42
python scripts/profile_dataset.py --dataset-root "Telepati 8.0 Datasets"
python scripts/run_final_training.py --config configs/final_model_config.yaml
python scripts/evaluate.py --weights runs/detect/final/final_model/weights/best.pt --split valid --conf-threshold 0.25
python scripts/run_error_analysis.py --weights runs/detect/final/final_model/weights/best.pt --split valid
```

### Artefak audit utama

| Berkas | Isi |
|---|---|
| `artifacts/audit/dataset_audit_report.md` | Audit forensik dataset mentah |
| `artifacts/audit/canonical_mapping_report.md` | Validasi pemetaan 11 kelas |
| `artifacts/audit/reproducibility_checklist.md` | Daftar periksa reproducibility |
| `artifacts/audit/block16_clean_reproduction_test.md` | Uji reproduksi lingkungan bersih |
| `artifacts/reports/dataset_profile.json` | Profiling dataset lengkap |
| `artifacts/reports/evaluation_valid.json` | Hasil evaluasi split validasi |
| `artifacts/reports/block13_error_analysis.json` | Analisis kesalahan |
| `artifacts/experiments/experiment_log.json` | 21 percobaan terkontrol |